# CLASSY-Breast — Breast MRI Morphology Workflow

CLASSY-Breast is a dataset-agnostic workflow for quantitative breast MRI morphology. Each MRI is processed completely before the next MRI starts:

1. optional metadata-based eligibility;
2. breast segmentation;
3. deformation assessment;
4. nipple annotation or reuse of existing manual nipple geometry;
5. three quadrant methods, numerical quality control, and persistent export.

All anatomical calculations use physical NIfTI RAS coordinates: **−X = patient left, +X = patient right, and +Z = superior**. Input MRI volumes, breast masks, and landmark files are expected to be in mutually consistent physical geometry before analysis.

The notebook is self-contained with respect to CLASSY-Breast logic. It does not include imaging data, trained weights, or third-party segmentation source code.


## License scope and third-party dependency

The CLASSY-Breast code contained in this notebook is governed by the repository `LICENSE` file (PolyForm Noncommercial 1.0.0).

**FGT-DUKE is an external third-party dependency and is explicitly outside the CLASSY-Breast license scope.** No FGT-DUKE source code or trained weights are redistributed in this notebook or repository. CLASSY-Breast only calls a separately installed copy through its public prediction interface.

For breast segmentation, this notebook uses the pretrained breast model and prediction code from the external repository by the Mazurowski Lab, referred to here as **FGT-DUKE**:

- Lew CO, Harouni M, Kirksey ER, et al. *A publicly available deep learning model and dataset for segmentation of breast, fibroglandular tissue, and vessels in breast MRI.* Scientific Reports. 2024;14:5383. doi:10.1038/s41598-024-54048-2.
- Official implementation: `https://github.com/mazurowski-lab/3D-Breast-FGT-and-Blood-Vessel-Segmentation`
- The external repository and its trained weights are licensed separately under **CC BY-NC-SA 4.0**. Users must obtain them separately and comply with those terms.

Recommended repository layout:

```text
CLASSY-Breast/
├── CLASSY-Breast.ipynb
├── LICENSE
├── README.md
├── source_data/
│   ├── images/<case_id>/*.nii.gz
│   └── metadata.xlsx                 # optional
├── external/
│   └── FGT-DUKE/                     # third-party; not distributed by CLASSY-Breast
├── masks_and_nipples/
└── results/
```


## 1. Environment configuration

This section defines project paths, imports required libraries, locates the external FGT-DUKE dependency, and checks the local 3D Slicer executable.

The public workflow uses generic input names and does not encode assumptions about the origin of the MRI collection. `metadata.xlsx` is optional; when it is absent, case identifiers are inferred from the subdirectories in `data/images/`.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

from scipy import ndimage
from scipy.spatial import cKDTree
from scipy.spatial.distance import jensenshannon

from IPython.display import display, HTML
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter

import plotly.graph_objects as go
from plotly.offline import plot
from skimage import measure


# ------------------------------------------------------------------
# Project paths
# ------------------------------------------------------------------

PROJECT = Path(
    os.environ.get(
        "CLASSY_BREAST_ROOT",
        Path.cwd(),
    )
).expanduser().resolve()

DATA_DIR = PROJECT / "source_data"
IMAGES_DIR = DATA_DIR / "images"
METADATA_FILE = DATA_DIR / "metadata.xlsx"
if not METADATA_FILE.exists():
    _alternate_metadata = DATA_DIR / "clinical_and_imaging_info.xlsx"
    if _alternate_metadata.exists():
        METADATA_FILE = _alternate_metadata

# ------------------------------------------------------------------
# External third-party dependency: FGT-DUKE
# ------------------------------------------------------------------
# FGT-DUKE is NOT distributed with CLASSY-Breast and is NOT covered
# by the CLASSY-Breast repository license. It must be obtained and
# installed separately under its own license (CC BY-NC-SA 4.0).

FGT_DUKE_REPO = Path(
    os.environ.get(
        "FGT_DUKE_REPO",
        PROJECT / "external" / "FGT-DUKE",
    )
).expanduser().resolve()

FGT_DUKE_WEIGHTS = Path(
    os.environ.get(
        "FGT_DUKE_WEIGHTS",
        FGT_DUKE_REPO / "trained_models" / "breast_model.pth",
    )
).expanduser().resolve()

FGT_DUKE_PREDICT = FGT_DUKE_REPO / "predict.py"

# ------------------------------------------------------------------
# 3D Slicer
# ------------------------------------------------------------------

_slicer_from_env = os.environ.get("SLICER_EXECUTABLE", "").strip()
if _slicer_from_env:
    SLICER = Path(_slicer_from_env).expanduser().resolve()
else:
    _slicer_on_path = shutil.which("Slicer")
    SLICER = (
        Path(_slicer_on_path).resolve()
        if _slicer_on_path
        else (
            Path.home()
            / "Applications"
            / "Slicer-5.12.3-linux-amd64"
            / "Slicer"
        )
    )

MASKS_DIR = PROJECT / "masks_and_nipples" / "masks"
DEFORMED_MASKS_DIR = MASKS_DIR / "deformed_masks"
NIPPLES_DIR = PROJECT / "masks_and_nipples" / "nipples"

STATUS_DIR = PROJECT / "status"
RESULTS_DIR = PROJECT / "results"

DEFORMATION_QC_DIR = RESULTS_DIR / "deformation_qc"
RANDOM_QC_DIR = RESULTS_DIR / "random_qc_html"
WORK_DIR = RESULTS_DIR / "_workflow_helpers"

WORKFLOW_STATE_FILE = STATUS_DIR / "workflow_case_status.csv"
DEFORMATION_RESULTS_FILE = RESULTS_DIR / "workflow_deformation_results.csv"
RESULTS_CSV = RESULTS_DIR / "workflow_results.csv"
RESULTS_XLSX = RESULTS_DIR / "CLASSY-Breast_results.xlsx"
QC_SELECTION_FILE = RESULTS_DIR / "random_qc_selection.csv"

for directory in [
    MASKS_DIR,
    DEFORMED_MASKS_DIR,
    NIPPLES_DIR,
    STATUS_DIR,
    RESULTS_DIR,
    DEFORMATION_QC_DIR,
    RANDOM_QC_DIR,
    WORK_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

assert IMAGES_DIR.exists(), (
    f"Input image directory not found: {IMAGES_DIR}"
)
assert FGT_DUKE_REPO.exists(), (
    "FGT-DUKE was not found. Install the external dependency or set "
    "the FGT_DUKE_REPO environment variable."
)
assert FGT_DUKE_PREDICT.exists(), (
    f"FGT-DUKE prediction script not found: {FGT_DUKE_PREDICT}"
)
assert FGT_DUKE_WEIGHTS.exists(), (
    f"FGT-DUKE breast weights not found: {FGT_DUKE_WEIGHTS}"
)
assert SLICER.exists(), (
    "3D Slicer executable not found. Set SLICER_EXECUTABLE if needed."
)

PYTHON = sys.executable

print("Project:", PROJECT)
print("Python:", PYTHON)
print("FGT-DUKE repository:", FGT_DUKE_REPO)
print("FGT-DUKE breast weights:", FGT_DUKE_WEIGHTS)
print("3D Slicer:", SLICER)


### Spatial geometry requirements

CLASSY-Breast does not perform source-specific anatomical corrections. Input MRI volumes are expected to have valid NIfTI qform/sform metadata, and any pre-existing masks or landmark files must already refer to the same physical space.

The workflow validates the spatial metadata before analysis and fails closed when MRI and mask geometry disagree. This keeps the public notebook portable and prevents hidden reorientation rules from being applied to data from an unrelated source.

All anatomical calculations use physical NIfTI RAS coordinates: **−X = patient left, +X = patient right, and +Z = superior**.


## 2. Metadata eligibility and workflow progress

Metadata is optional. If `source_data/metadata.xlsx` is present, the workflow uses a `case_id` column when available, or falls back to `patient_id`. If the file is absent, case identifiers are inferred from the folders in `data/images/`.

The following optional metadata fields are recognized when present:

- `bilateral_mri`: cases explicitly marked as non-bilateral are excluded;
- `bilateral_breast_cancer`: non-zero values are excluded;
- `multifocal_cancer`: non-zero values are excluded.

No source-specific inclusion or exclusion rule is encoded in the public notebook.

A case is considered *processed* when this workflow has reached a terminal state (`DONE` or `EXCLUDED`). Paused or failed cases remain in the *remaining* count.


In [ ]:
def _numeric_flag(value, default=0):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return default

    converted = pd.to_numeric(
        pd.Series([value]),
        errors="coerce",
    ).iloc[0]

    if pd.isna(converted):
        return default

    return converted


if METADATA_FILE.exists():
    metadata = pd.read_excel(METADATA_FILE).copy()

    if "case_id" in metadata.columns:
        metadata["case_id"] = metadata["case_id"].astype(str)
    elif "patient_id" in metadata.columns:
        metadata["case_id"] = metadata["patient_id"].astype(str)
    else:
        raise RuntimeError(
            "metadata.xlsx must contain either 'case_id' or 'patient_id'."
        )
else:
    case_ids = sorted(
        path.name
        for path in IMAGES_DIR.iterdir()
        if path.is_dir()
    )

    if not case_ids:
        raise RuntimeError(
            f"No case folders were found in {IMAGES_DIR}."
        )

    metadata = pd.DataFrame({"case_id": case_ids})


metadata["case_id"] = metadata["case_id"].astype(str)


def metadata_exclusion_reasons(row):
    reasons = []

    if "bilateral_mri" in metadata.columns:
        bilateral_mri = (
            _numeric_flag(
                row.get("bilateral_mri", 1),
                default=1,
            ) == 1
        )
        if not bilateral_mri:
            reasons.append("unilateral_MRI")

    if "bilateral_breast_cancer" in metadata.columns:
        if _numeric_flag(row.get("bilateral_breast_cancer", 0)) != 0:
            reasons.append("bilateral_breast_cancer")

    if "multifocal_cancer" in metadata.columns:
        if _numeric_flag(row.get("multifocal_cancer", 0)) != 0:
            reasons.append("multifocal_breast_cancer")

    return ";".join(reasons)


metadata["metadata_exclusion_reason"] = metadata.apply(
    metadata_exclusion_reasons,
    axis=1,
)

metadata["metadata_excluded"] = (
    metadata["metadata_exclusion_reason"].ne("")
)

eligible_metadata = (
    metadata.loc[
        ~metadata["metadata_excluded"]
    ]
    .copy()
    .sort_values(["case_id"])
    .reset_index(drop=True)
)


def safe_read_csv(path):
    path = Path(path)

    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame()

    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def build_workflow_state():
    base = metadata[
        [
            "case_id",
            "metadata_excluded",
            "metadata_exclusion_reason",
        ]
    ].copy()

    base["pipeline_status"] = np.where(
        base["metadata_excluded"],
        "EXCLUDED_METADATA",
        "PENDING",
    )

    base["excluded"] = base["metadata_excluded"].astype(bool)
    base["exclusion_reason"] = (
        base["metadata_exclusion_reason"]
        .fillna("")
        .astype(str)
    )

    base["mask_path"] = ""
    base["nipple_path"] = ""
    base["deformation_class"] = ""
    base["deformation_source"] = ""
    base["processed_order"] = np.nan
    base["last_error"] = ""

    previous = safe_read_csv(
        WORKFLOW_STATE_FILE
    )

    if previous.empty:
        return base

    preserve = [
        "pipeline_status",
        "excluded",
        "exclusion_reason",
        "mask_path",
        "nipple_path",
        "deformation_class",
        "deformation_source",
        "processed_order",
        "last_error",
    ]

    available = [
        column
        for column in preserve
        if column in previous.columns
    ]

    if not available:
        return base

    merged = base.merge(
        previous[["case_id"] + available],
        on="case_id",
        how="left",
        suffixes=("", "_saved"),
    )

    for column in available:
        saved = f"{column}_saved"

        if saved not in merged.columns:
            continue

        use_saved = merged[saved].notna()
        merged.loc[use_saved, column] = (
            merged.loc[use_saved, saved]
        )

        merged = merged.drop(
            columns=[saved]
        )

    return merged


state_df = build_workflow_state()

eligible_ids = set(
    eligible_metadata["case_id"].astype(str)
)

eligible_state = state_df[
    state_df["case_id"].astype(str).isin(
        eligible_ids
    )
].copy()

terminal_statuses = {
    "DONE",
    "EXCLUDED",
}

processed = int(
    eligible_state["pipeline_status"]
    .isin(terminal_statuses)
    .sum()
)

total_eligible = len(eligible_state)
remaining = total_eligible - processed

print(f"Metadata-eligible cases: {total_eligible}")
print(f"Already processed:       {processed}")
print(f"Remaining:               {remaining}")


## 3. Breast segmentation

Breast segmentation is performed through the external **FGT-DUKE** dependency described above. The CLASSY-Breast notebook prepares the input volume, invokes the external prediction script, restores the prediction to native voxel order, and applies binary morphology and connected-component cleanup.

The external FGT-DUKE code and trained weights are not embedded in this notebook. The interface below only passes files and command-line arguments to a separately installed copy.

If a valid mask already exists in the standard mask directory, it is reused. Otherwise, segmentation is computed when the case reaches this step.


In [ ]:
def validate_nifti_geometry(image, case_id, role):
    """Fail closed on inconsistent spatial metadata; never guess anatomy."""
    if len(image.shape) < 3 or any(n < 2 for n in image.shape[:3]):
        raise RuntimeError(f"{case_id}: invalid {role} shape {image.shape}.")

    affine = np.asarray(image.affine, dtype=float)

    if affine.shape != (4, 4) or not np.all(np.isfinite(affine)):
        raise RuntimeError(f"{case_id}: invalid {role} affine.")

    determinant = np.linalg.det(affine[:3, :3])
    if not np.isfinite(determinant) or abs(determinant) < 1e-8:
        raise RuntimeError(f"{case_id}: singular {role} affine.")

    if not np.all(np.asarray(image.header.get_zooms()[:3]) > 0):
        raise RuntimeError(f"{case_id}: invalid {role} voxel spacing.")

    qcode = int(image.header["qform_code"])
    scode = int(image.header["sform_code"])

    if qcode > 0 and scode > 0:
        if not np.allclose(
            image.get_qform(),
            image.get_sform(),
            atol=1e-3,
            rtol=1e-5,
        ):
            raise RuntimeError(
                f"{case_id}: {role} qform/sform conflict; review before processing."
            )

    if qcode == 0 and scode == 0:
        raise RuntimeError(
            f"{case_id}: {role} has no explicit qform/sform orientation."
        )

    return affine


def _nifti_with_affine(data, source_image, affine):
    """Create a NIfTI image with synchronized qform and sform."""
    header = source_image.header.copy()
    image = nib.Nifti1Image(data, affine, header=header)

    qcode = int(source_image.header["qform_code"])
    scode = int(source_image.header["sform_code"])

    image.set_qform(affine, code=qcode if qcode > 0 else 1)
    image.set_sform(affine, code=scode if scode > 0 else 1)

    return image


def load_mri(case_id):
    """Load one MRI after validating its physical NIfTI geometry."""
    image = nib.load(find_image(case_id))
    validate_nifti_geometry(image, case_id, "MRI")
    return image


def slicer_mri_path(case_id):
    """Return the original MRI path after validating its geometry."""
    path = find_image(case_id)
    image = nib.load(path)
    validate_nifti_geometry(image, case_id, "MRI")
    return path


def validate_mask_matches_mri(case_id, mask_image):
    mri = load_mri(case_id)
    validate_nifti_geometry(mask_image, case_id, "mask")

    if tuple(mri.shape[:3]) != tuple(mask_image.shape[:3]):
        raise RuntimeError(f"{case_id}: MRI/mask shapes disagree.")

    if not np.allclose(
        mri.affine,
        mask_image.affine,
        atol=1e-3,
        rtol=1e-5,
    ):
        raise RuntimeError(
            f"{case_id}: mask affine does not match MRI geometry."
        )


def find_image(case_id):
    case_dir = IMAGES_DIR / case_id

    preferred = (
        case_dir
        / f"{case_id.lower()}_0000.nii.gz"
    )

    if preferred.exists():
        return preferred

    files = sorted(
        case_dir.glob("*.nii.gz")
    )

    if not files:
        raise FileNotFoundError(
            f"No NIfTI file found for {case_id}: {case_dir}"
        )

    return files[0]


def to_network_orientation(image):
    image = np.transpose(
        image,
        (1, 0, 2),
    )

    image = np.rot90(
        image,
        2,
        axes=(0, 1),
    )

    return image.copy()


def from_network_orientation(mask):
    mask = np.rot90(
        mask,
        2,
        axes=(0, 1),
    )

    mask = np.transpose(
        mask,
        (1, 0, 2),
    )

    return mask.copy()


def normalize_for_network(image):
    values = image[
        np.isfinite(image)
    ]

    low, high = np.percentile(
        values,
        [0.1, 99.9],
    )

    if high <= low:
        raise RuntimeError(
            "Invalid MRI intensity range."
        )

    normalized = (
        image.astype(np.float32)
        - float(low)
    ) / float(high - low)

    normalized = np.clip(
        normalized,
        0.0,
        1.0,
    )

    standard_deviation = float(
        normalized.std()
    )

    if standard_deviation == 0:
        raise RuntimeError(
            "MRI standard deviation is zero."
        )

    return (
        (
            normalized
            - float(normalized.mean())
        )
        / standard_deviation
    ).astype(np.float32)


def clean_mask(prediction):
    mask = (
        np.asarray(prediction)
        > 0.5
    )

    structure = (
        ndimage.generate_binary_structure(
            3,
            1,
        )
    )

    mask = ndimage.binary_opening(
        mask,
        structure=structure,
        iterations=3,
    )

    labeled, number = ndimage.label(
        mask
    )

    if number == 0:
        return mask.astype(np.uint8)

    sizes = ndimage.sum(
        mask,
        labeled,
        range(1, number + 1),
    )

    largest = float(
        np.max(sizes)
    )

    output = np.zeros_like(
        mask,
        dtype=np.uint8,
    )

    for label_id, size in enumerate(
        sizes,
        start=1,
    ):
        if float(size) >= 0.10 * largest:
            output[
                labeled == label_id
            ] = 1

    for k in range(
        output.shape[2]
    ):
        output[:, :, k] = (
            ndimage.binary_fill_holes(
                output[:, :, k]
            )
        )

    return output.astype(np.uint8)


# ==================================================================
# EXTERNAL DEPENDENCY INTERFACE — FGT-DUKE
# This wrapper is CLASSY-Breast code. It invokes, but does not copy,
# the separately installed third-party FGT-DUKE implementation.
# ==================================================================

def run_fgt_duke_segmentation(
    case_id,
    network_image,
):
    temporary_root = (
        RESULTS_DIR
        / "_segmentation_tmp"
    )

    input_dir = (
        temporary_root
        / "input"
    )

    prediction_dir = (
        temporary_root
        / "pred"
    )

    shutil.rmtree(
        temporary_root,
        ignore_errors=True,
    )

    input_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    prediction_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    np.save(
        input_dir
        / f"{case_id}.npy",
        network_image.astype(
            np.float32
        ),
    )

    result = subprocess.run(
        [
            PYTHON,
            "predict.py",
            "--target-tissue",
            "breast",
            "--image",
            str(input_dir),
            "--save-masks-dir",
            str(prediction_dir),
            "--model-save-path",
            str(FGT_DUKE_WEIGHTS),
        ],
        cwd=str(FGT_DUKE_REPO),
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"{case_id}: segmentation inference failed.\n"
            f"{result.stderr[-2000:]}"
        )

    prediction_path = (
        prediction_dir
        / f"{case_id}.npy"
    )

    if not prediction_path.exists():
        candidates = list(
            prediction_dir.glob("*.npy")
        )

        if len(candidates) != 1:
            raise RuntimeError(
                f"{case_id}: prediction file was not found."
            )

        prediction_path = candidates[0]

    prediction = np.load(
        prediction_path
    )

    shutil.rmtree(
        temporary_root,
        ignore_errors=True,
    )

    return clean_mask(
        prediction
    )



def save_mask(
    case_id,
    mask_native,
    nifti_image,
):
    case_dir = (
        MASKS_DIR
        / case_id
    )

    case_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    path = (
        case_dir
        / f"{case_id}_mask.nii.gz"
    )
    output_affine = np.asarray(nifti_image.affine, dtype=float)

    header = (
        nifti_image.header.copy()
    )

    header.set_data_dtype(
        np.uint8
    )

    output = _nifti_with_affine(
        mask_native.astype(np.uint8),
        nifti_image,
        output_affine,
    )
    output.header.set_data_dtype(np.uint8)

    nib.save(
        output,
        path,
    )

    return path



def segment_case(
    case_id,
    overwrite=False,
):
    output_path = (
        MASKS_DIR
        / case_id
        / f"{case_id}_mask.nii.gz"
    )

    if (
        output_path.exists()
        and not overwrite
    ):
        validate_mask_matches_mri(case_id, nib.load(output_path))
        print(
            f"{case_id}: existing breast mask reused."
        )
        return output_path

    image_path = find_image(
        case_id
    )

    nifti_image = nib.load(
        image_path
    )

    validate_nifti_geometry(nifti_image, case_id, "MRI")

    # Segmentation is performed on the native voxel array.
    native_image = np.asarray(
        nifti_image.dataobj,
        dtype=np.float32,
    ).squeeze()

    if native_image.ndim != 3:
        raise RuntimeError(
            f"{case_id}: expected a 3D volume, "
            f"found {native_image.shape}."
        )

    network_image = (
        to_network_orientation(
            native_image
        )
    )

    network_image = (
        normalize_for_network(
            network_image
        )
    )

    network_mask = run_fgt_duke_segmentation(
        case_id,
        network_image,
    )

    native_mask = (
        from_network_orientation(
            network_mask
        )
    )

    if (
        native_mask.shape
        != native_image.shape
    ):
        raise RuntimeError(
            f"{case_id}: mask shape "
            f"{native_mask.shape} does not match MRI shape "
            f"{native_image.shape}."
        )

    return save_mask(
        case_id,
        native_mask,
        nifti_image,
    )




## 4. Deformation detection

After segmentation, each case is assessed for anterior breast deformation.

The detector searches the most informative axial slices, identifies long near-horizontal anterior boundary segments in each breast, quantifies their length, angle and fitting error, and combines these measurements with bilateral symmetry.

Three automatic outcomes are possible:

- **PASS** — no manual decision is required;
- **EXCLUDED** — the case is automatically excluded for deformation;
- **REVIEW** — the most suspicious slice for each breast is displayed and a manual decision is required.

### Visual quality control

The most suspicious slice is displayed for **every case**, not only for `REVIEW` cases. The visualization intentionally contains only the MRI and the detected line; the segmentation mask is not overlaid.

For `REVIEW` cases:

- `P` / `p` = PASS;
- `E` / `e` = EXCLUDED;
- `S` / `s` = open the complete MRI in 3D Slicer for inspection.

When `S` is selected, 3D Slicer opens the MRI **without the segmentation mask**. After Slicer is closed, the notebook returns to the same classification prompt.


In [ ]:

TOP_FRACTION = 0.05
DEFORMATION_AREA_THRESHOLD = 0.85

CENTER_RANGE = (0.35, 0.65)
VERT_TOL = 5
MIN_GAP_LINE = 20
SIDE_MARGIN = 3
SMOOTH_SIGMA = 1.5

ANTERIOR_BAND_FRACTION = 0.32
MIN_SEGMENT_MM = 18.0
MAX_ORTHOGONAL_RMSE_MM = 2.0
MAX_ORTHOGONAL_P95_MM = 3.8
MAX_ABS_ANGLE_DEG = 35.0
ROBUST_KEEP_PERCENTILE = 90.0
ROBUST_ITERATIONS = 3

LENGTH_LOW_MM = 35.0
LENGTH_HIGH_MM = 110.0
ANGLE_GOOD_DEG = 5.0
ANGLE_BAD_DEG = 30.0
RMSE_GOOD_MM = 0.8
RMSE_BAD_MM = 2.2

SUSPICIOUS_BREAST_SCORE = 60.0
STRONG_BREAST_SCORE = 70.0
MIN_BILATERAL_SYMMETRY = 58.0
REVIEW_SCORE = 55.0

def largest_component(mask):
    labels, n = ndimage.label(mask)

    if n == 0:
        return mask.astype(bool)

    counts = np.bincount(labels.ravel())
    label = 1 + np.argmax(counts[1:])

    return labels == label


def to_deformation_orientation(x):
    """Transform native NIfTI [I,J,K] into the detector orientation [J,I,K] with a 180° axial rotation."""
    return to_network_orientation(x)


def deformation_spacing(nii):
    """Return voxel spacing after swapping the first two axes."""
    zi = nii.header.get_zooms()[:3]
    return np.array(
        [float(zi[1]), float(zi[0]), float(zi[2])],
        dtype=float,
    )


def interbreast_box(slice_mask):
    _, width = slice_mask.shape

    anterior = np.full(
        width,
        np.nan,
        dtype=float,
    )

    for x in range(width):
        ys = np.flatnonzero(
            slice_mask[:, x]
        )

        if len(ys):
            anterior[x] = ys.min()

    valid = np.isfinite(anterior)

    if valid.sum() < 20:
        return None

    xx = np.arange(width)

    curve = np.interp(
        xx,
        xx[valid],
        anterior[valid],
    )

    curve = ndimage.gaussian_filter1d(
        curve,
        SMOOTH_SIGMA,
    )

    start = int(CENTER_RANGE[0] * width)
    end = int(CENTER_RANGE[1] * width)

    best = None

    for xl in range(start, end):
        for xr in range(
            xl + MIN_GAP_LINE,
            end,
        ):
            segment = curve[xl:xr + 1]

            if (
                segment.max()
                - segment.min()
                <= VERT_TOL
            ):
                score = (
                    xr - xl,
                    float(np.median(segment)),
                )

                if best is None or score > best[0]:
                    best = (
                        score,
                        xl,
                        xr,
                    )

    if best is None:
        return None

    _, xl, xr = best

    xl = max(
        0,
        xl - SIDE_MARGIN,
    )

    xr = min(
        width - 1,
        xr + SIDE_MARGIN,
    )

    return (xl + xr) / 2.0


def deformation_interbreast_plane(mask):
    areas = mask.sum(axis=(0, 1))

    if areas.max() == 0:
        raise RuntimeError("Empty mask.")

    selected = np.flatnonzero(
        areas
        >= DEFORMATION_AREA_THRESHOLD
        * areas.max()
    )

    mids = []

    for z in selected:
        midpoint = interbreast_box(
            mask[:, :, z]
        )

        if midpoint is not None:
            mids.append(midpoint)

    if mids:
        return float(np.mean(mids))

    columns = np.flatnonzero(
        mask.any(axis=(0, 2))
    )

    if len(columns) < 2:
        raise RuntimeError(
            "The two breasts could not be separated."
        )

    return float(
        (columns[0] + columns[-1]) / 2.0
    )


def split_breasts_detector(mask, xsplit):
    x = np.arange(
        mask.shape[1]
    )[None, :, None]

    side_a = largest_component(
        mask & (x < xsplit)
    )

    side_b = largest_component(
        mask & (x >= xsplit)
    )

    # Detector-side labels are kept internally for consistency.
    return {
        "L": side_a,
        "R": side_b,
    }


def top_slices(mask):
    """Select the top 5% of non-empty slices by breast-mask area."""
    areas = mask.sum(axis=(0, 1))
    valid = np.flatnonzero(areas > 0)

    if len(valid) == 0:
        return np.array([], dtype=int), areas

    n = max(
        1,
        int(np.ceil(
            len(valid) * TOP_FRACTION
        )),
    )

    ranked = valid[
        np.argsort(
            areas[valid]
        )[::-1]
    ]

    return ranked[:n], areas


def anterior_profile(mask_slice):
    width = mask_slice.shape[1]

    profile = np.full(
        width,
        np.nan,
        dtype=float,
    )

    for x in range(width):
        ys = np.flatnonzero(
            mask_slice[:, x]
        )

        if len(ys):
            profile[x] = ys.min()

    return profile


def breast_depth(mask_slice):
    ys = np.flatnonzero(
        mask_slice.any(axis=1)
    )

    if len(ys) < 2:
        return 0.0

    return float(
        ys[-1] - ys[0] + 1
    )


def candidate_runs(mask_slice):
    profile = anterior_profile(
        mask_slice
    )

    valid = np.isfinite(profile)

    if valid.sum() < 8:
        return [], profile

    min_y = float(
        np.nanmin(profile)
    )

    depth = breast_depth(
        mask_slice
    )

    if depth <= 0:
        return [], profile

    candidate = (
        valid
        & (
            profile
            <= min_y
            + ANTERIOR_BAND_FRACTION
            * depth
        )
    )

    labels, n = ndimage.label(
        candidate.astype(np.uint8)
    )

    runs = []

    for label in range(1, n + 1):
        indices = np.flatnonzero(
            labels == label
        )

        if len(indices) >= 4:
            runs.append(indices)

    return runs, profile


def fit_tls(points_mm):
    center = points_mm.mean(axis=0)
    centered = points_mm - center

    _, _, vh = np.linalg.svd(
        centered,
        full_matrices=False,
    )

    direction = vh[0]
    direction = (
        direction
        / np.linalg.norm(direction)
    )

    normal = np.array(
        [-direction[1], direction[0]],
        dtype=float,
    )

    residuals = np.abs(
        centered @ normal
    )

    return (
        center,
        direction,
        residuals,
    )


def robust_line(points_mm):
    current = np.arange(
        len(points_mm)
    )

    for _ in range(
        ROBUST_ITERATIONS
    ):
        center, direction, residuals = (
            fit_tls(
                points_mm[current]
            )
        )

        cutoff = np.percentile(
            residuals,
            ROBUST_KEEP_PERCENTILE,
        )

        new_current = current[
            residuals <= cutoff
        ]

        if (
            len(new_current) < 4
            or len(new_current)
            == len(current)
        ):
            break

        current = new_current

    center, direction, _ = fit_tls(
        points_mm[current]
    )

    all_centered = (
        points_mm - center
    )

    normal = np.array(
        [-direction[1], direction[0]],
        dtype=float,
    )

    residuals = np.abs(
        all_centered @ normal
    )

    projections = (
        all_centered @ direction
    )

    p0 = (
        center
        + projections.min()
        * direction
    )

    p1 = (
        center
        + projections.max()
        * direction
    )

    length = float(
        np.linalg.norm(p1 - p0)
    )

    rmse = float(
        np.sqrt(
            np.mean(
                residuals ** 2
            )
        )
    )

    p95 = float(
        np.percentile(
            residuals,
            95,
        )
    )

    angle = abs(
        np.degrees(
            np.arctan2(
                direction[1],
                direction[0],
            )
        )
    )

    angle = min(
        angle,
        abs(180.0 - angle),
    )

    return {
        "length_mm": length,
        "rmse_mm": rmse,
        "p95_mm": p95,
        "angle_deg": angle,
        "p0_mm": p0,
        "p1_mm": p1,
    }


def best_linear_segment(
    mask_slice,
    spacing,
):
    runs, profile = candidate_runs(
        mask_slice
    )

    if not runs:
        return None

    row_spacing = float(spacing[0])
    col_spacing = float(spacing[1])

    best = None

    for run in runs:
        x_values = run
        y_values = profile[run]

        points_mm = np.column_stack([
            x_values * col_spacing,
            y_values * row_spacing,
        ])

        n = len(points_mm)

        for start in range(
            0,
            n - 3,
        ):
            for end in range(
                n,
                start + 3,
                -1,
            ):
                subset = points_mm[
                    start:end
                ]

                endpoint_distance = float(
                    np.linalg.norm(
                        subset[-1]
                        - subset[0]
                    )
                )

                if (
                    endpoint_distance
                    < MIN_SEGMENT_MM
                ):
                    break

                fit = robust_line(
                    subset
                )

                if (
                    fit["angle_deg"]
                    > MAX_ABS_ANGLE_DEG
                ):
                    continue

                if (
                    fit["rmse_mm"]
                    > MAX_ORTHOGONAL_RMSE_MM
                ):
                    continue

                if (
                    fit["p95_mm"]
                    > MAX_ORTHOGONAL_P95_MM
                ):
                    continue

                candidate = {
                    **fit,
                    "x_start":
                        int(
                            x_values[start]
                        ),
                    "x_end":
                        int(
                            x_values[
                                end - 1
                            ]
                        ),
                }

                if (
                    best is None
                    or (
                        candidate[
                            "length_mm"
                        ]
                        > best[
                            "length_mm"
                        ]
                    )
                    or (
                        abs(
                            candidate[
                                "length_mm"
                            ]
                            - best[
                                "length_mm"
                            ]
                        ) < 1e-6
                        and (
                            candidate[
                                "rmse_mm"
                            ]
                            < best[
                                "rmse_mm"
                            ]
                        )
                    )
                ):
                    best = candidate

                break

    return best


def best_breast_detection(
    breast_mask,
    spacing,
    selected_slices,
):
    candidates = []

    z_valid = np.flatnonzero(
        breast_mask.sum(
            axis=(0, 1)
        ) > 0
    )

    if len(z_valid) == 0:
        return None

    z_min = int(z_valid.min())
    z_max = int(z_valid.max())

    z_span = max(
        1,
        z_max - z_min,
    )

    for z in selected_slices:
        if not breast_mask[
            :, :, z
        ].any():
            continue

        line = best_linear_segment(
            breast_mask[:, :, z],
            spacing,
        )

        if line is None:
            continue

        candidates.append({
            "z": int(z),
            "z_normalized":
                float(z - z_min)
                / float(z_span),
            "line": line,
        })

    if not candidates:
        return None

    return max(
        candidates,
        key=lambda item: (
            item[
                "line"
            ][
                "length_mm"
            ],
            -item[
                "line"
            ][
                "rmse_mm"
            ],
        ),
    )


def scale_increasing(
    value,
    low,
    high,
):
    if high <= low:
        return 0.0

    return float(
        np.clip(
            (value - low)
            / (high - low),
            0.0,
            1.0,
        ) * 100.0
    )


def scale_decreasing(
    value,
    good,
    bad,
):
    if bad <= good:
        return 0.0

    return float(
        np.clip(
            (bad - value)
            / (bad - good),
            0.0,
            1.0,
        ) * 100.0
    )


def breast_deformation_score(
    detection,
):
    if detection is None:
        return {
            "length_score": 0.0,
            "angle_score": 0.0,
            "fit_score": 0.0,
            "score": 0.0,
        }

    line = detection["line"]

    length_score = scale_increasing(
        line["length_mm"],
        LENGTH_LOW_MM,
        LENGTH_HIGH_MM,
    )

    angle_score = scale_decreasing(
        line["angle_deg"],
        ANGLE_GOOD_DEG,
        ANGLE_BAD_DEG,
    )

    fit_score = scale_decreasing(
        line["rmse_mm"],
        RMSE_GOOD_MM,
        RMSE_BAD_MM,
    )

    score = (
        length_score
        + angle_score
        + fit_score
    ) / 3.0

    return {
        "length_score":
            length_score,
        "angle_score":
            angle_score,
        "fit_score":
            fit_score,
        "score":
            score,
    }


def deformation_symmetry_score(
    left_detection,
    right_detection,
):
    if (
        left_detection is None
        or right_detection is None
    ):
        return {
            "length_symmetry": 0.0,
            "angle_symmetry": 0.0,
            "slice_symmetry": 0.0,
            "score": 0.0,
        }

    left_line = (
        left_detection["line"]
    )

    right_line = (
        right_detection["line"]
    )

    max_length = max(
        left_line["length_mm"],
        right_line["length_mm"],
        1.0,
    )

    length_symmetry = (
        1.0
        - abs(
            left_line["length_mm"]
            - right_line["length_mm"]
        )
        / max_length
    ) * 100.0

    angle_symmetry = (
        1.0
        - min(
            abs(
                left_line["angle_deg"]
                - right_line["angle_deg"]
            )
            / 25.0,
            1.0,
        )
    ) * 100.0

    slice_symmetry = (
        1.0
        - min(
            abs(
                left_detection[
                    "z_normalized"
                ]
                - right_detection[
                    "z_normalized"
                ]
            )
            / 0.20,
            1.0,
        )
    ) * 100.0

    score = (
        length_symmetry
        + angle_symmetry
        + slice_symmetry
    ) / 3.0

    return {
        "length_symmetry":
            float(np.clip(
                length_symmetry,
                0.0,
                100.0,
            )),
        "angle_symmetry":
            float(np.clip(
                angle_symmetry,
                0.0,
                100.0,
            )),
        "slice_symmetry":
            float(np.clip(
                slice_symmetry,
                0.0,
                100.0,
            )),
        "score":
            float(np.clip(
                score,
                0.0,
                100.0,
            )),
    }


def classify_deformation(
    left_score,
    right_score,
    symmetry,
):
    max_score = max(
        left_score,
        right_score,
    )

    min_score = min(
        left_score,
        right_score,
    )

    both_suspicious = (
        left_score
        >= SUSPICIOUS_BREAST_SCORE
        and right_score
        >= SUSPICIOUS_BREAST_SCORE
    )

    if (
        both_suspicious
        and symmetry
        >= MIN_BILATERAL_SYMMETRY
    ):
        return "EXCLUDED"

    if (
        max_score
        >= STRONG_BREAST_SCORE
        or max_score
        >= REVIEW_SCORE
        or (
            min_score
            >= SUSPICIOUS_BREAST_SCORE
            and symmetry
            < MIN_BILATERAL_SYMMETRY
        )
    ):
        return "REVIEW"

    return "PASS"


def analyse_deformation_case(
    case_id,
    mask_native,
    image_native,
    nii,
):
    """Run deformation detection in the detector coordinate system."""
    mask = (
        to_deformation_orientation(
            mask_native.astype(np.uint8)
        ) > 0
    )

    image = (
        to_deformation_orientation(
            image_native
        )
    )

    spacing = deformation_spacing(
        nii
    )

    xsplit = (
        deformation_interbreast_plane(
            mask
        )
    )

    breasts = split_breasts_detector(
        mask,
        xsplit,
    )

    selected_slices, areas = (
        top_slices(mask)
    )

    if len(selected_slices) == 0:
        raise RuntimeError(
            "No usable slices were found."
        )

    detections = {
        side:
            best_breast_detection(
                breast,
                spacing,
                selected_slices,
            )
        for side, breast
        in breasts.items()
    }

    scores = {
        side:
            breast_deformation_score(
                detections[side]
            )
        for side in ("L", "R")
    }

    symmetry = (
        deformation_symmetry_score(
            detections["L"],
            detections["R"],
        )
    )

    initial_class = (
        classify_deformation(
            scores["L"]["score"],
            scores["R"]["score"],
            symmetry["score"],
        )
    )

    return {
        "mask_detector": mask,
        "image_detector": image,
        "spacing": spacing,
        "xsplit": xsplit,
        "selected_slices":
            selected_slices,
        "detections":
            detections,
        "scores":
            scores,
        "symmetry":
            symmetry,
        "initial_class":
            initial_class,
    }


In [ ]:

def line_pixels(
    line,
    spacing,
):
    row_spacing = float(
        spacing[0]
    )

    column_spacing = float(
        spacing[1]
    )

    p0 = line["p0_mm"]
    p1 = line["p1_mm"]

    x = np.array([
        p0[0] / column_spacing,
        p1[0] / column_spacing,
    ])

    y = np.array([
        p0[1] / row_spacing,
        p1[1] / row_spacing,
    ])

    return x, y


def image_limits(image):
    values = image[
        np.isfinite(image)
    ]

    if len(values) == 0:
        return None, None

    return tuple(
        np.percentile(
            values,
            [1, 99],
        )
    )


def plot_detection_panel(
    ax,
    image,
    detection,
    score,
    spacing,
    side,
    fallback_z=None,
):
    """
    Show MRI only, with the detected suspicious line.
    No segmentation contour is displayed.
    """

    if detection is None:
        if fallback_z is None:
            ax.text(
                0.5,
                0.5,
                f"Breast {side}\nNo valid suspicious segment",
                ha="center",
                va="center",
            )
            ax.axis("off")
            return

        z = int(fallback_z)
        img = image[:, :, z]
        low, high = image_limits(img)

        ax.imshow(
            img,
            cmap="gray",
            vmin=low,
            vmax=high,
        )

        ax.set_title(
            f"Breast {side} — no valid line | z={z}",
            fontsize=9,
        )

        ax.axis("off")
        return

    z = int(
        detection["z"]
    )

    img = image[:, :, z]

    low, high = image_limits(
        img
    )

    ax.imshow(
        img,
        cmap="gray",
        vmin=low,
        vmax=high,
    )

    line = detection["line"]

    x, y = line_pixels(
        line,
        spacing,
    )

    ax.plot(
        x,
        y,
        linewidth=4.0,
        solid_capstyle="round",
    )

    ax.scatter(
        x,
        y,
        s=20,
    )

    ax.set_title(
        (
            f"Breast {side} — score={score:.1f} | z={z}\n"
            f"length={line['length_mm']:.1f} mm | "
            f"angle={line['angle_deg']:.1f}° | "
            f"RMSE={line['rmse_mm']:.2f} mm | "
            f"P95={line['p95_mm']:.2f} mm"
        ),
        fontsize=9,
    )

    ax.axis("off")


def make_deformation_figure(
    case_id,
    result,
    classification,
    source,
):
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 7),
    )

    selected = result[
        "selected_slices"
    ]

    fallback_z = (
        int(selected[0])
        if len(selected)
        else None
    )

    for ax, side in zip(
        axes,
        ("L", "R"),
    ):
        plot_detection_panel(
            ax=ax,
            image=result[
                "image_detector"
            ],
            detection=result[
                "detections"
            ][side],
            score=result[
                "scores"
            ][side]["score"],
            spacing=result[
                "spacing"
            ],
            side=side,
            fallback_z=fallback_z,
        )

    fig.suptitle(
        (
            f"{case_id} — {classification}\n"
            f"Bilateral symmetry score = "
            f"{result['symmetry']['score']:.1f} | "
            f"decision source = {source}"
        ),
        fontsize=13,
    )

    fig.tight_layout()

    return fig


def display_deformation_qc(
    case_id,
    result,
    classification,
    source,
):
    fig = make_deformation_figure(
        case_id=case_id,
        result=result,
        classification=classification,
        source=source,
    )

    plt.show()


def save_deformation_qc(
    case_id,
    result,
    classification,
    source,
):
    fig = make_deformation_figure(
        case_id=case_id,
        result=result,
        classification=classification,
        source=source,
    )

    path = (
        DEFORMATION_QC_DIR
        / f"{case_id}_deformation.png"
    )

    fig.savefig(
        path,
        dpi=180,
        bbox_inches="tight",
    )

    plt.close(fig)

    return path


def move_mask_to_deformed(
    case_id,
):
    source_dir = (
        MASKS_DIR
        / case_id
    )

    target_dir = (
        DEFORMED_MASKS_DIR
        / case_id
    )

    target_dir.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if target_dir.exists():
        shutil.rmtree(
            target_dir
        )

    if source_dir.exists():
        shutil.move(
            str(source_dir),
            str(target_dir),
        )

    mask_path = (
        target_dir
        / f"{case_id}_mask.nii.gz"
    )

    if not mask_path.exists():
        raise FileNotFoundError(
            f"{case_id}: mask not found after relocation."
        )

    return mask_path


# ------------------------------------------------------------------
# 3D Slicer helper for manual deformation review.
# MRI only: the breast mask is deliberately not loaded.
# ------------------------------------------------------------------

REVIEW_HELPER = (
    WORK_DIR
    / "inspect_mri_slicer.py"
)

REVIEW_HELPER.write_text(
    r"""
import sys
from pathlib import Path

import qt
import slicer

CASE = sys.argv[-2]
IMAGE = Path(sys.argv[-1])

volume = slicer.util.loadVolume(
    str(IMAGE)
)

if volume is None:
    raise RuntimeError(
        f"Could not open MRI: {IMAGE}"
    )

volume.SetName(CASE)

try:
    layout_manager = slicer.app.layoutManager()
    layout_manager.setLayout(
        slicer.vtkMRMLLayoutNode.SlicerLayoutFourUpView
    )
    slicer.util.resetSliceViews()
except Exception as exc:
    print("Layout warning:", exc)

try:
    logic = slicer.modules.volumerendering.logic()
    display = logic.GetFirstVolumeRenderingDisplayNode(
        volume
    )

    if display is None:
        display = logic.CreateDefaultVolumeRenderingNodes(
            volume
        )

    display.SetVisibility(True)

    preset = logic.GetPresetByName(
        "MR-Default"
    )

    if (
        preset is not None
        and display.GetVolumePropertyNode() is not None
    ):
        display.GetVolumePropertyNode().Copy(
            preset
        )
except Exception as exc:
    print("Volume-rendering warning:", exc)

dialog = qt.QDialog(
    slicer.util.mainWindow()
)

dialog.setWindowTitle(
    f"CLASSY-Breast — deformation review — {CASE}"
)

dialog.setModal(False)

dialog.setWindowFlags(
    dialog.windowFlags()
    | qt.Qt.WindowStaysOnTopHint
)

layout = qt.QVBoxLayout(dialog)

label = qt.QLabel(
    f"<b>{CASE}</b><br><br>"
    "Inspect the complete MRI freely in all slices.<br>"
    "No segmentation mask is displayed.<br><br>"
    "When inspection is complete, click the button below."
)

label.setWordWrap(True)
layout.addWidget(label)

button = qt.QPushButton(
    "Close 3D Slicer and return to classification"
)

layout.addWidget(button)


def finish():
    dialog.close()
    qt.QTimer.singleShot(
        250,
        slicer.util.exit,
    )


button.connect(
    "clicked()",
    finish,
)

dialog.resize(
    620,
    220,
)

slicer._classyBreastReviewDialog = dialog

dialog.show()
dialog.raise_()
dialog.activateWindow()
""",
    encoding="utf-8",
)


def _clean_slicer_environment():
    environment = os.environ.copy()

    for key in [
        "PYTHONHOME",
        "PYTHONPATH",
        "QT_PLUGIN_PATH",
        "QT_QPA_PLATFORM_PLUGIN_PATH",
        "QML2_IMPORT_PATH",
        "LD_PRELOAD",
        "QT_XCB_GL_INTEGRATION",
        "QT_OPENGL",
    ]:
        environment.pop(
            key,
            None,
        )

    return environment


def open_review_in_slicer(
    case_id,
):
    image_path = slicer_mri_path(
        case_id
    )

    print(
        f"Opening {case_id} in 3D Slicer without a mask..."
    )

    result = subprocess.run(
        [
            str(SLICER),
            "--python-script",
            str(REVIEW_HELPER),
            case_id,
            str(image_path),
        ],
        env=_clean_slicer_environment(),
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"{case_id}: 3D Slicer exited "
            f"with code {result.returncode}."
        )


def ask_review_decision(
    case_id,
):
    while True:
        answer = input(
            f"{case_id}: "
            "[P]ASS | [E]XCLUDED | "
            "[S] open complete MRI in 3D Slicer: "
        ).strip().upper()

        if answer in {
            "P",
            "PASS",
        }:
            return "PASS"

        if answer in {
            "E",
            "EXCLUDED",
        }:
            return "EXCLUDED"

        if answer in {
            "S",
            "SLICER",
        }:
            open_review_in_slicer(
                case_id
            )
            continue

        print(
            "Invalid input. Use P, E or S."
        )


## 5. Nipple annotation

Nipple coordinates are required only for cases that pass deformation assessment.

If a valid annotation containing exactly two control points already exists, it is reused automatically. Otherwise, the MRI is opened in 3D Slicer and the user is asked to mark:

1. the patient's **left nipple**;
2. the patient's **right nipple**.

The Slicer window remains fully interactive during annotation. The annotation can also be paused and resumed on another day.

If fewer than two nipples can be identified, the user may explicitly exclude the case for `nipple_not_annotatable`.


In [ ]:

NIPPLE_WORK_DIR = (
    WORK_DIR
    / "nipple_annotation"
)

NIPPLE_WORK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def nipple_annotation_path(
    case_id,
):
    case_dir = (
        NIPPLES_DIR
        / case_id
    )

    case_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    return (
        case_dir
        / f"{case_id}_nipples.mrk.json"
    )


def valid_nipple_annotation(
    path,
):
    path = Path(path)

    if not path.exists():
        return False

    try:
        data = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

        points = []
        for markup in data.get("markups", []):
            system = str(markup.get("coordinateSystem", "LPS")).upper()
            if system not in ("LPS", "RAS", "0", "1"):
                return False
            for point in markup.get("controlPoints", []):
                p = np.asarray(point["position"], dtype=float)
                if p.shape != (3,) or not np.all(np.isfinite(p)):
                    return False
                x_ras = -p[0] if system in ("LPS", "1") else p[0]
                points.append(float(x_ras))
        return len(points) == 2 and abs(points[0] - points[1]) >= 10.0

    except Exception:
        return False


NIPPLE_HELPER = (
    NIPPLE_WORK_DIR
    / "annotate_nipples_slicer.py"
)

NIPPLE_HELPER.write_text(
    r"""
import sys
from pathlib import Path

import qt
import slicer

CASE = sys.argv[-4]
IMAGE = Path(sys.argv[-3])
ANNOTATION = Path(sys.argv[-2])
DECISION = Path(sys.argv[-1])

ANNOTATION.parent.mkdir(
    parents=True,
    exist_ok=True,
)

DECISION.parent.mkdir(
    parents=True,
    exist_ok=True,
)

DECISION.unlink(
    missing_ok=True,
)

volume = slicer.util.loadVolume(
    str(IMAGE)
)

if volume is None:
    raise RuntimeError(
        f"Could not open MRI: {IMAGE}"
    )

volume.SetName(CASE)

try:
    layout_manager = slicer.app.layoutManager()
    layout_manager.setLayout(
        slicer.vtkMRMLLayoutNode.SlicerLayoutFourUpView
    )
    slicer.util.resetSliceViews()
except Exception as exc:
    print("Layout warning:", exc)

try:
    logic = slicer.modules.volumerendering.logic()

    display = (
        logic.GetFirstVolumeRenderingDisplayNode(
            volume
        )
    )

    if display is None:
        display = (
            logic.CreateDefaultVolumeRenderingNodes(
                volume
            )
        )

    display.SetVisibility(True)

    preset = logic.GetPresetByName(
        "MR-Default"
    )

    if (
        preset is not None
        and display.GetVolumePropertyNode() is not None
    ):
        display.GetVolumePropertyNode().Copy(
            preset
        )
except Exception as exc:
    print("Volume-rendering warning:", exc)

nipples = slicer.mrmlScene.AddNewNodeByClass(
    "vtkMRMLMarkupsFiducialNode",
    "Nipples",
)

try:
    slicer.util.selectModule("Markups")
except Exception:
    pass

selection_node = slicer.mrmlScene.GetNodeByID(
    "vtkMRMLSelectionNodeSingleton"
)

interaction_node = slicer.mrmlScene.GetNodeByID(
    "vtkMRMLInteractionNodeSingleton"
)

selection_node.SetReferenceActivePlaceNodeClassName(
    "vtkMRMLMarkupsFiducialNode"
)

selection_node.SetActivePlaceNodeID(
    nipples.GetID()
)

interaction_node.SetPlaceModePersistence(1)

interaction_node.SetCurrentInteractionMode(
    interaction_node.Place
)

dialog = qt.QDialog(
    slicer.util.mainWindow()
)

dialog.setWindowTitle(
    f"CLASSY-Breast — nipple annotation — {CASE}"
)

dialog.setModal(False)

dialog.setWindowFlags(
    dialog.windowFlags()
    | qt.Qt.WindowStaysOnTopHint
)

layout = qt.QVBoxLayout(dialog)

info = qt.QLabel(
    f"<b>{CASE}</b><br><br>"
    "Place exactly two points:<br><br>"
    "1st = <b>patient's left nipple</b><br>"
    "2nd = <b>patient's right nipple</b><br><br>"
    "You may navigate, zoom and rotate the MRI while this window is open."
)

info.setWordWrap(True)
layout.addWidget(info)

status_label = qt.QLabel()
layout.addWidget(status_label)

save_button = qt.QPushButton(
    "Save and continue"
)

pause_button = qt.QPushButton(
    "Pause and continue another day"
)

layout.addWidget(save_button)
layout.addWidget(pause_button)


def number_defined():
    try:
        return int(
            nipples.GetNumberOfDefinedControlPoints()
        )
    except Exception:
        return int(
            nipples.GetNumberOfControlPoints()
        )


def update_status(*args):
    number = number_defined()
    # Relabel immediately; also enforced again immediately before save.
    for idx, name in enumerate(("Nipple_L", "Nipple_R")):
        if idx < number and nipples.GetNthControlPointLabel(idx) != name:
            nipples.SetNthControlPointLabel(idx, name)

    if number == 0:
        text = "0/2 points placed"
    elif number == 1:
        text = "1/2 points placed"
    elif number == 2:
        text = "2/2 points placed — ready to save"
    else:
        text = (
            f"{number}/2 points placed — "
            "remove the extra points"
        )

    status_label.setText(text)


def finish():
    try:
        interaction_node.SetCurrentInteractionMode(
            interaction_node.ViewTransform
        )
    except Exception:
        pass

    dialog.close()

    qt.QTimer.singleShot(
        250,
        slicer.util.exit,
    )


def save():
    number = number_defined()

    if number == 2:
        # Markups positions are in Slicer world RAS regardless of the
        # coordinate system chosen when the .mrk.json is exported.
        positions = []
        for idx in range(2):
            ras = [0.0, 0.0, 0.0]
            nipples.GetNthControlPointPositionWorld(idx, ras)
            if not all(__import__("math").isfinite(float(v)) for v in ras):
                qt.QMessageBox.warning(dialog, "Invalid points",
                    "Nipple coordinates must be finite.")
                return
            positions.append(ras)

        # Automatically associate labels with anatomical laterality if
        # the two points were placed in the opposite order.
        if abs(positions[0][0] - positions[1][0]) < 10.0:
            qt.QMessageBox.warning(dialog, "Ambiguous laterality",
                "The nipples are too close in RAS X to identify L/R. "
                "Please review the points.")
            return
        left_idx = 0 if positions[0][0] < positions[1][0] else 1
        right_idx = 1 - left_idx
        nipples.SetNthControlPointLabel(left_idx, "Nipple_L")
        nipples.SetNthControlPointLabel(right_idx, "Nipple_R")
        confirm = qt.QMessageBox.question(
            dialog, "Confirm patient's left and right nipples",
            f"Patient LEFT: point {left_idx + 1} "
            f"(RAS X = {positions[left_idx][0]:.2f} mm)\n"
            f"Patient RIGHT: point {right_idx + 1} "
            f"(RAS X = {positions[right_idx][0]:.2f} mm)\n\n"
            "Are both points on the correct anatomical nipples?",
            qt.QMessageBox.Yes | qt.QMessageBox.No,
            qt.QMessageBox.No,
        )
        if confirm != qt.QMessageBox.Yes:
            return
        if ANNOTATION.exists():
            qt.QMessageBox.warning(dialog, "Original annotation preserved",
                "An annotation file already exists. It was NOT overwritten. "
                "Pause and review the original annotation.")
            return
        success = slicer.util.saveNode(
            nipples,
            str(ANNOTATION),
        )

        if not success:
            qt.QMessageBox.critical(
                dialog,
                "Error",
                "The nipple annotation could not be saved.",
            )
            return

        DECISION.write_text(
            "ANNOTATED\n",
            encoding="utf-8",
        )

        finish()
        return

    if number < 2:
        answer = qt.QMessageBox.question(
            dialog,
            "Exclude MRI?",
            (
                f"Only {number} of 2 nipples were marked.\n\n"
                "Exclude this MRI because the nipples cannot "
                "be annotated reliably?"
            ),
            qt.QMessageBox.Yes
            | qt.QMessageBox.No,
            qt.QMessageBox.No,
        )

        if answer == qt.QMessageBox.Yes:
            # Never delete an existing original markup during exclusion.
            DECISION.write_text(
                "EXCLUDED\n",
                encoding="utf-8",
            )

            finish()

        return

    qt.QMessageBox.warning(
        dialog,
        "Too many points",
        (
            f"{number} points were placed. "
            "Exactly two are required."
        ),
    )


def pause():
    DECISION.write_text(
        "PAUSE\n",
        encoding="utf-8",
    )

    finish()


save_button.connect(
    "clicked()",
    save,
)

pause_button.connect(
    "clicked()",
    pause,
)

try:
    nipples.AddObserver(
        nipples.PointModifiedEvent,
        update_status,
    )

    nipples.AddObserver(
        nipples.PointPositionDefinedEvent,
        update_status,
    )
except Exception:
    pass

update_status()

dialog.resize(
    580,
    300,
)

slicer._classyBreastNippleDialog = dialog
slicer._classyBreastNippleNode = nipples

dialog.show()
dialog.raise_()
dialog.activateWindow()
""",
    encoding="utf-8",
)


def annotate_nipples(
    case_id,
):
    annotation = nipple_annotation_path(
        case_id
    )

    if valid_nipple_annotation(
        annotation
    ):
        print(
            f"{case_id}: existing nipple annotation reused."
        )

        return {
            "status": "ANNOTATED",
            "annotation": str(annotation),
        }

    image_path = slicer_mri_path(
        case_id
    )

    decision_path = (
        NIPPLE_WORK_DIR
        / f"{case_id}_decision.txt"
    )

    decision_path.unlink(
        missing_ok=True,
    )

    print(
        f"Opening {case_id} in 3D Slicer for nipple annotation..."
    )

    result = subprocess.run(
        [
            str(SLICER),
            "--python-script",
            str(NIPPLE_HELPER),
            case_id,
            str(image_path),
            str(annotation),
            str(decision_path),
        ],
        env=_clean_slicer_environment(),
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"{case_id}: 3D Slicer exited "
            f"with code {result.returncode}."
        )

    if not decision_path.exists():
        raise RuntimeError(
            f"{case_id}: 3D Slicer was closed "
            "without saving an annotation, pausing, "
            "or confirming exclusion."
        )

    decision = (
        decision_path.read_text(
            encoding="utf-8"
        )
        .strip()
        .upper()
    )

    if decision == "ANNOTATED":
        if not valid_nipple_annotation(
            annotation
        ):
            raise RuntimeError(
                f"{case_id}: the saved annotation "
                "does not contain exactly two points."
            )

        return {
            "status": "ANNOTATED",
            "annotation": str(annotation),
        }

    if decision == "EXCLUDED":
        return {
            "status": "EXCLUDED",
            "annotation": "",
        }

    if decision == "PAUSE":
        return {
            "status": "PAUSE",
            "annotation": "",
        }

    raise RuntimeError(
        f"{case_id}: unknown 3D Slicer decision: {decision!r}"
    )


## 6. Breast geometry and quadrant definitions

The segmented bilateral breast mask is separated into left and right breasts in physical RAS coordinates.

Four quadrants are reported for each breast:

- superior-medial;
- superior-lateral;
- inferior-medial;
- inferior-lateral.

Three geometric definitions are evaluated.

### Method 1 — nipple + anatomical axes

Both dividing planes pass through the nipple. The medial–lateral plane is defined by a normal aligned with the anatomical left–right axis, and the superior–inferior plane is defined by a normal aligned with the superior–inferior axis.

### Method 2 — nipple-to-centroid geometry

The breast centroid is computed in physical space. The nipple-to-centroid direction defines the depth axis; cross products with the anatomical superior direction generate orthogonal medial–lateral and superior–inferior normals. The two quadrant-dividing planes pass through the centroid.

### Method 3 — nipple-to-posterior chest-wall geometry

A posterior chest-wall reference is estimated from the posterior breast envelope. A posterior reference plane and a nipple-related plane are combined geometrically, and the nipple is projected onto the resulting intermediate plane. The nipple-to-projection direction defines breast depth, from which orthogonal medial–lateral and superior–inferior normals are constructed.


In [ ]:

AREA_THRESHOLD = 0.85


def interbreast_plane(
    mask,
):
    mask = np.asarray(
        mask
    ).astype(bool)

    areas = np.array(
        [
            mask[:, :, z].sum()
            for z
            in range(
                mask.shape[2]
            )
        ],
        dtype=float,
    )

    if areas.max() == 0:
        raise RuntimeError(
            "Empty breast mask."
        )

    selected = np.flatnonzero(
        areas
        >= AREA_THRESHOLD
        * areas.max()
    )

    midpoints = []

    for z in selected:
        coordinates = np.argwhere(
            mask[:, :, z]
        )

        if len(coordinates) == 0:
            continue

        x = coordinates[:, 0]

        midpoints.append(
            0.5
            * (
                float(x.min())
                + float(x.max())
            )
        )

    if not midpoints:
        raise RuntimeError(
            "The inter-breast plane could not be estimated."
        )

    return float(
        np.mean(midpoints)
    )


def component_centroid_ras(
    mask,
    affine,
):
    ijk = np.argwhere(
        mask
    )

    if len(ijk) == 0:
        return None

    homogeneous = np.column_stack(
        [
            ijk,
            np.ones(
                len(ijk)
            ),
        ]
    )

    ras = (
        homogeneous
        @ affine.T
    )[:, :3]

    return ras.mean(
        axis=0
    )


def split_breasts_native(
    mask,
    affine,
):
    split = interbreast_plane(
        mask
    )

    ii = np.arange(
        mask.shape[0]
    )[:, None, None]

    part_a = (
        mask
        & (ii <= split)
    )

    part_b = (
        mask
        & (ii > split)
    )

    ras_a = component_centroid_ras(
        part_a,
        affine,
    )

    ras_b = component_centroid_ras(
        part_b,
        affine,
    )

    if (
        ras_a is None
        or ras_b is None
    ):
        raise RuntimeError(
            "The breast mask could not be separated into two breasts."
        )

    # NIfTI affine uses physical RAS coordinates: -X = patient's L,
    # +X = patient's R. Compare the breast centroids, not voxel order.
    if ras_a[0] < ras_b[0]:
        return {"L": part_a, "R": part_b}, split

    return {"R": part_a, "L": part_b}, split



def load_mask(
    case_id,
):
    path = (
        MASKS_DIR
        / case_id
        / f"{case_id}_mask.nii.gz"
    )

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    nifti_image = nib.load(
        path
    )

    validate_mask_matches_mri(case_id, nifti_image)

    mask = (
        np.asarray(
            nifti_image.dataobj
        ).squeeze()
        > 0
    )

    return mask, nifti_image



def load_nipples_ras(
    case_id,
):
    """Load the two nipple landmarks in canonical physical RAS coordinates.

    Landmark files are expected to use the same physical geometry as the MRI
    and breast mask. No coordinate migration or silent anatomical correction
    is performed when landmarks are loaded.
    """
    path = (
        NIPPLES_DIR
        / case_id
        / f"{case_id}_nipples.mrk.json"
    )

    if not path.exists():
        raise FileNotFoundError(path)

    data = json.loads(
        path.read_text(encoding="utf-8")
    )

    points = []

    for markup in data.get("markups", []):
        coordinate_system = str(
            markup.get("coordinateSystem", "LPS")
        ).strip().upper()

        if coordinate_system in ("1", "LPS"):
            coordinate_system = "LPS"
        elif coordinate_system in ("0", "RAS"):
            coordinate_system = "RAS"
        else:
            raise RuntimeError(
                f"{case_id}: unsupported markup coordinate system "
                f"{coordinate_system!r}."
            )

        for control_point in markup.get("controlPoints", []):
            point = np.asarray(
                control_point["position"],
                dtype=float,
            )

            if point.shape != (3,) or not np.all(np.isfinite(point)):
                raise RuntimeError(
                    f"{case_id}: invalid nipple coordinate {point}."
                )

            if coordinate_system == "LPS":
                point = np.array(
                    [-point[0], -point[1], point[2]],
                    dtype=float,
                )

            points.append(
                (
                    str(control_point.get("label", "")).strip(),
                    point,
                )
            )

    if len(points) != 2:
        raise RuntimeError(
            f"{case_id}: expected exactly two nipple points, "
            f"found {len(points)}."
        )

    # In physical RAS, smaller X is patient left and larger X is patient right.
    points = sorted(points, key=lambda item: item[1][0])

    if points[1][1][0] - points[0][1][0] < 10.0:
        raise RuntimeError(
            f"{case_id}: nipple left/right positions are ambiguous."
        )

    for expected, (label, _) in zip(("L", "R"), points):
        normalized = label.upper()
        if normalized in ("NIPPLE_L", "NIPPLE_R"):
            if normalized != f"NIPPLE_{expected}":
                raise RuntimeError(
                    f"{case_id}: saved nipple label {label!r} disagrees "
                    "with anatomical RAS X. Review the annotation."
                )

    return {"L": points[0][1], "R": points[1][1]}


def ijk_to_ras(
    points_ijk,
    affine,
):
    points = np.asarray(
        points_ijk,
        dtype=float,
    )

    if points.ndim == 1:
        points = points[None, :]

    homogeneous = np.column_stack(
        [
            points,
            np.ones(
                len(points)
            ),
        ]
    )

    return (
        homogeneous
        @ affine.T
    )[:, :3]


def breast_points_ras(
    breast_mask,
    affine,
):
    ijk = np.argwhere(
        breast_mask
    )

    return ijk_to_ras(
        ijk,
        affine,
    )


def nipple_to_mask_distance_mm(
    breast_mask,
    affine,
    nipple_ras,
):
    eroded = ndimage.binary_erosion(
        breast_mask
    )

    surface = (
        breast_mask
        & ~eroded
    )

    ijk = np.argwhere(
        surface
    )

    if len(ijk) == 0:
        return np.nan

    ras = ijk_to_ras(
        ijk,
        affine,
    )

    tree = cKDTree(
        ras
    )

    distance, _ = tree.query(
        np.asarray(
            nipple_ras
        ),
        k=1,
    )

    return float(
        distance
    )


def quadrant_counts_from_planes(
    breast_mask,
    affine,
    origin_ras,
    medial_normal,
    superior_normal,
):
    points = np.argwhere(
        breast_mask
    )

    if len(points) == 0:
        raise RuntimeError(
            "Empty breast mask."
        )

    ras = ijk_to_ras(
        points,
        affine,
    )

    relative = (
        ras
        - np.asarray(
            origin_ras
        )[None, :]
    )

    medial = (
        relative
        @ np.asarray(
            medial_normal
        )
    ) >= 0

    superior = (
        relative
        @ np.asarray(
            superior_normal
        )
    ) >= 0

    selections = {
        "superior_medial":
            superior & medial,

        "superior_lateral":
            superior & ~medial,

        "inferior_medial":
            ~superior & medial,

        "inferior_lateral":
            ~superior & ~medial,
    }

    voxel_cm3 = (
        abs(
            np.linalg.det(
                affine[:3, :3]
            )
        )
        / 1000.0
    )

    total = len(
        points
    )

    result = {}

    for name, selection in (
        selections.items()
    ):
        number = int(
            selection.sum()
        )

        result[name] = {
            "voxels":
                number,

            "volume_cm3":
                number
                * voxel_cm3,

            "percentage":
                100.0
                * number
                / total,
        }

    return result


In [ ]:
def method1(case_id):
    mask, nii = load_mask(case_id)
    nipples = load_nipples_ras(case_id)

    breasts, xsplit = split_breasts_native(mask, nii.affine)

    rows = []

    for side in ("L", "R"):
        # L medial points +RAS X; R medial points -RAS X.
        medial_normal = (
            np.array([1.0, 0.0, 0.0])
            if side == "L"
            else np.array([-1.0, 0.0, 0.0])
        )

        superior_normal = np.array([0.0, 0.0, 1.0])

        quadrants = quadrant_counts_from_planes(
            breasts[side],
            nii.affine,
            nipples[side],
            medial_normal,
            superior_normal,
        )

        row = {
            "case_id": case_id,
            "breast": side,
            "method": "M1",
        }

        for q, vals in quadrants.items():
            row[f"{q}_cm3"] = vals["volume_cm3"]
            row[f"{q}_pct"] = vals["percentage"]

        row["total_cm3"] = sum(
            vals["volume_cm3"]
            for vals in quadrants.values()
        )

        rows.append(row)

    return rows


In [ ]:
def normalized(v):
    v = np.asarray(v, dtype=float)
    n = np.linalg.norm(v)

    if n == 0:
        raise RuntimeError("Zero-length vector.")

    return v / n


def method2_axes(nipple, centroid, side):
    depth_axis = normalized(centroid - nipple)

    superior_ref = np.array([0.0, 0.0, 1.0])

    medial = np.cross(superior_ref, depth_axis)

    if np.linalg.norm(medial) < 1e-8:
        medial = np.array([1.0, 0.0, 0.0])

    medial = normalized(medial)

    desired_medial = (
        np.array([1.0, 0.0, 0.0])
        if side == "L"
        else np.array([-1.0, 0.0, 0.0])
    )

    if np.dot(medial, desired_medial) < 0:
        medial = -medial

    superior = normalized(
        np.cross(depth_axis, medial)
    )

    if np.dot(superior, superior_ref) < 0:
        superior = -superior

    return depth_axis, medial, superior


def method2(case_id):
    mask, nii = load_mask(case_id)
    nipples = load_nipples_ras(case_id)

    breasts, xsplit = split_breasts_native(mask, nii.affine)

    rows = []

    for side in ("L", "R"):
        ras = breast_points_ras(
            breasts[side],
            nii.affine,
        )

        centroid = ras.mean(axis=0)

        depth, medial, superior = method2_axes(
            nipples[side],
            centroid,
            side,
        )

        quadrants = quadrant_counts_from_planes(
            breasts[side],
            nii.affine,
            centroid,
            medial,
            superior,
        )

        row = {
            "case_id": case_id,
            "breast": side,
            "method": "M2",
        }

        for q, vals in quadrants.items():
            row[f"{q}_cm3"] = vals["volume_cm3"]
            row[f"{q}_pct"] = vals["percentage"]

        row["total_cm3"] = sum(
            vals["volume_cm3"]
            for vals in quadrants.values()
        )

        rows.append(row)

    return rows


In [ ]:
def posterior_surface_points(breast_mask, affine):
    """Estimate the posterior envelope in physical RAS space by retaining the point with the smallest Y coordinate for each (I,K) location."""
    ijk = np.argwhere(breast_mask)

    if len(ijk) == 0:
        raise RuntimeError("Empty breast mask.")

    ras = ijk_to_ras(ijk, affine)

    # Group by (I,K) and retain the most posterior point in each group.
    keys = ijk[:, [0, 2]]

    order = np.lexsort((keys[:, 1], keys[:, 0]))
    ijk = ijk[order]
    ras = ras[order]
    keys = keys[order]

    out = []

    start = 0
    while start < len(keys):
        end = start + 1

        while end < len(keys) and np.array_equal(keys[end], keys[start]):
            end += 1

        group = ras[start:end]
        out.append(group[np.argmin(group[:, 1])])

        start = end

    pts = np.asarray(out)

    # Restrict the posterior surface to the central 80% in X and Z.
    xlo, xhi = np.quantile(pts[:, 0], [0.10, 0.90])
    zlo, zhi = np.quantile(pts[:, 2], [0.10, 0.90])

    keep = (
        (pts[:, 0] >= xlo)
        & (pts[:, 0] <= xhi)
        & (pts[:, 2] >= zlo)
        & (pts[:, 2] <= zhi)
    )

    pts = pts[keep]

    if len(pts) < 20:
        raise RuntimeError("Too few posterior surface points.")

    return pts


def filter_posterior_points(points, k=5, mad_factor=5.0):
    pts = np.asarray(points, dtype=float)

    tree = cKDTree(pts)
    dists, _ = tree.query(pts, k=min(k + 1, len(pts)))

    knn = dists[:, -1]
    med = np.median(knn)
    mad = np.median(np.abs(knn - med))

    if mad > 0:
        keep = knn <= med + mad_factor * mad
        pts = pts[keep]

    return pts


def fit_plane_svd(points):
    pts = np.asarray(points, dtype=float)

    center = pts.mean(axis=0)
    _, _, vh = np.linalg.svd(
        pts - center,
        full_matrices=False,
    )

    normal = normalized(vh[-1])

    return center, normal


def project_to_plane(point, plane_point, plane_normal):
    p = np.asarray(point, dtype=float)
    q = np.asarray(plane_point, dtype=float)
    n = normalized(plane_normal)

    return p - np.dot(p - q, n) * n


def angular_bisector_plane(
    plane1_point,
    plane1_normal,
    plane2_point,
    plane2_normal,
):
    n1 = normalized(plane1_normal)
    n2 = normalized(plane2_normal)

    if np.dot(n1, n2) < 0:
        n2 = -n2

    bisector = n1 + n2

    if np.linalg.norm(bisector) < 1e-8:
        bisector = n1

    bisector = normalized(bisector)

    midpoint = 0.5 * (
        np.asarray(plane1_point)
        + np.asarray(plane2_point)
    )

    return midpoint, bisector


def method3_geometry(breast_mask, affine, nipple, side):
    posterior = posterior_surface_points(
        breast_mask,
        affine,
    )

    posterior = filter_posterior_points(
        posterior,
        k=5,
        mad_factor=5.0,
    )

    global_point, global_normal = fit_plane_svd(
        posterior
    )

    # C is the valid posterior point nearest to the nipple.
    tree = cKDTree(posterior)
    _, idx = tree.query(nipple, k=1)
    C = posterior[int(idx)]

    NC = normalized(C - nipple)

    # The second plane passes through C and is perpendicular to the nipple-to-C direction.
    second_point = C
    second_normal = NC

    mid_point, mid_normal = angular_bisector_plane(
        global_point,
        global_normal,
        second_point,
        second_normal,
    )

    P = project_to_plane(
        nipple,
        mid_point,
        mid_normal,
    )

    depth_axis = normalized(P - nipple)

    superior_ref = np.array([0.0, 0.0, 1.0])
    medial = np.cross(superior_ref, depth_axis)

    if np.linalg.norm(medial) < 1e-8:
        medial = np.array([1.0, 0.0, 0.0])

    medial = normalized(medial)

    desired_medial = (
        np.array([1.0, 0.0, 0.0])
        if side == "L"
        else np.array([-1.0, 0.0, 0.0])
    )

    if np.dot(medial, desired_medial) < 0:
        medial = -medial

    superior = normalized(
        np.cross(depth_axis, medial)
    )

    if np.dot(superior, superior_ref) < 0:
        superior = -superior

    return {
        "C": C,
        "P": P,
        "depth_axis": depth_axis,
        "medial_normal": medial,
        "superior_normal": superior,
        "posterior_plane_point": mid_point,
        "posterior_plane_normal": mid_normal,
    }


def method3(case_id):
    mask, nii = load_mask(case_id)
    nipples = load_nipples_ras(case_id)

    breasts, xsplit = split_breasts_native(mask, nii.affine)

    rows = []

    for side in ("L", "R"):
        geom = method3_geometry(
            breasts[side],
            nii.affine,
            nipples[side],
            side,
        )

        # Quadrant division uses the depth reference defined by the nipple-to-P geometry.
        origin = geom["P"]

        quadrants = quadrant_counts_from_planes(
            breasts[side],
            nii.affine,
            origin,
            geom["medial_normal"],
            geom["superior_normal"],
        )

        row = {
            "case_id": case_id,
            "breast": side,
            "method": "M3",
            "NP_mm": float(
                np.linalg.norm(
                    geom["P"] - nipples[side]
                )
            ),
        }

        for q, vals in quadrants.items():
            row[f"{q}_cm3"] = vals["volume_cm3"]
            row[f"{q}_pct"] = vals["percentage"]

        row["total_cm3"] = sum(
            vals["volume_cm3"]
            for vals in quadrants.values()
        )

        rows.append(row)

    return rows


In [ ]:
QUADRANTS = [
    "superior_medial",
    "superior_lateral",
    "inferior_medial",
    "inferior_lateral",
]


def qc_method_row(row, tolerance=1e-6):
    volumes = np.array(
        [row[f"{q}_cm3"] for q in QUADRANTS],
        dtype=float,
    )

    percentages = np.array(
        [row[f"{q}_pct"] for q in QUADRANTS],
        dtype=float,
    )

    return {
        "volume_sum_ok":
            abs(volumes.sum() - row["total_cm3"]) <= tolerance,

        "percentage_sum":
            float(percentages.sum()),

        "percentage_sum_ok":
            abs(percentages.sum() - 100.0) <= 1e-6,
    }


def nipple_qc(case_id):
    mask, nii = load_mask(case_id)
    nipples = load_nipples_ras(case_id)
    breasts, _ = split_breasts_native(mask, nii.affine)

    return {
        side: nipple_to_mask_distance_mm(
            breasts[side],
            nii.affine,
            nipples[side],
        )
        for side in ("L", "R")
    }


## 7. Persistent results and random interactive quality control

After each case reaches a terminal state, the workflow immediately updates:

- the workflow state CSV;
- deformation measurements;
- quadrant measurements;
- the Excel workbook.

The Excel workbook contains separate sheets for case status, Methods 1–3, method comparisons, numerical QC, deformation results, and random interactive QC selections.

### Random HTML QC

For every complete block of **10 successfully processed (`DONE`) MRIs**, one case is selected randomly and reproducibly.

For the selected case, an interactive 3D HTML report is:

1. saved in `results/random_qc_html/`;
2. recorded in the Excel workbook;
3. **displayed directly in the notebook** when it is generated.

The HTML allows switching between Methods 1, 2 and 3 and inspecting the four breast quadrants interactively.


In [ ]:

QUADRANTS = [
    "superior_medial",
    "superior_lateral",
    "inferior_medial",
    "inferior_lateral",
]

QC_RANDOM_SEED = 20260924

QUADRANT_COLORS = {
    "superior_medial": "#d62728",
    "superior_lateral": "#ffbf00",
    "inferior_medial": "#1f77b4",
    "inferior_lateral": "#2ca02c",
}


def qc_method_row(
    row,
    tolerance=1e-6,
):
    volumes = np.array(
        [
            row[f"{quadrant}_cm3"]
            for quadrant
            in QUADRANTS
        ],
        dtype=float,
    )

    percentages = np.array(
        [
            row[f"{quadrant}_pct"]
            for quadrant
            in QUADRANTS
        ],
        dtype=float,
    )

    return {
        "volume_sum_ok":
            abs(
                volumes.sum()
                - row["total_cm3"]
            )
            <= tolerance,

        "percentage_sum":
            float(
                percentages.sum()
            ),

        "percentage_sum_ok":
            abs(
                percentages.sum()
                - 100.0
            )
            <= 1e-6,
    }


def nipple_qc(
    case_id,
):
    mask, nifti_image = load_mask(
        case_id
    )

    nipples = load_nipples_ras(
        case_id
    )

    breasts, _ = split_breasts_native(
        mask,
        nifti_image.affine,
    )

    return {
        side:
            nipple_to_mask_distance_mm(
                breasts[side],
                nifti_image.affine,
                nipples[side],
            )
        for side
        in ("L", "R")
    }


def atomic_csv(
    dataframe,
    path,
):
    path = Path(path)

    temporary = path.with_suffix(
        path.suffix
        + ".tmp"
    )

    dataframe.to_csv(
        temporary,
        index=False,
    )

    temporary.replace(
        path
    )


def comparison_table(
    results_df,
):
    rows = []

    if (
        results_df.empty
        or "method"
        not in results_df.columns
    ):
        return pd.DataFrame()

    for (
        case_id,
        breast,
    ), group in results_df.groupby(
        [
            "case_id",
            "breast",
        ]
    ):
        vectors = {}

        for method in (
            "M1",
            "M2",
            "M3",
        ):
            subset = group[
                group["method"].eq(
                    method
                )
            ]

            if len(subset) != 1:
                continue

            row = subset.iloc[0]

            vectors[method] = np.array(
                [
                    row[
                        f"{quadrant}_pct"
                    ]
                    for quadrant
                    in QUADRANTS
                ],
                dtype=float,
            )

        for first, second in [
            ("M1", "M2"),
            ("M1", "M3"),
            ("M2", "M3"),
        ]:
            if (
                first not in vectors
                or second not in vectors
            ):
                continue

            a = vectors[first]
            b = vectors[second]

            difference = np.abs(
                a - b
            )

            p = a / a.sum()
            q = b / b.sum()

            rows.append({
                "case_id":
                    case_id,

                "breast":
                    breast,

                "comparison":
                    f"{first}-{second}",

                "MAE_pp":
                    float(
                        difference.mean()
                    ),

                "max_abs_pp":
                    float(
                        difference.max()
                    ),

                "L1_pp":
                    float(
                        difference.sum()
                    ),

                "JS_distance":
                    float(
                        jensenshannon(
                            p,
                            q,
                        )
                    ),
            })

    return pd.DataFrame(
        rows
    )


def round_to_significant_digits(value, digits=6):
    """Round a finite number for Excel presentation only."""
    if value is None or isinstance(value, (bool, np.bool_)):
        return value
    if not isinstance(value, (int, float, np.integer, np.floating)):
        return value
    value = float(value)
    if not np.isfinite(value) or value == 0:
        return value
    if isinstance(value, (int, np.integer)):
        return int(value)
    places = digits - 1 - int(np.floor(np.log10(abs(value))))
    return round(value, places)


def excel_presentation_copy(dataframe, digits=6):
    """Copy a table and round numeric Excel values to significant digits."""
    table = dataframe.copy()
    for column in table.columns:
        if pd.api.types.is_numeric_dtype(table[column]):
            if pd.api.types.is_bool_dtype(table[column]):
                continue
            table[column] = table[column].map(
                lambda value: round_to_significant_digits(value, digits)
                if pd.notna(value) else value
            )
    return table


def format_excel_file(path):
    """Format the workbook for readable six-significant-digit results."""
    workbook = load_workbook(path)

    for worksheet in workbook.worksheets:
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions

        for cell in worksheet[1]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True,
            )

        for column_index, column in enumerate(
            worksheet.iter_cols(),
            start=1,
        ):
            maximum_length = 0

            for cell in column:
                value = cell.value
                if value is None:
                    continue
                if isinstance(value, float):
                    cell.number_format = "General"
                maximum_length = max(maximum_length, len(str(value)))

            # Deliberately generous widths so rounded numbers never show ###.
            width = min(max(maximum_length + 6, 24), 72)
            worksheet.column_dimensions[
                get_column_letter(column_index)
            ].width = width

    workbook.save(path)


def save_excel_snapshot(
    state_df,
    results_df,
    deformation_df,
):
    comparison_df = comparison_table(
        results_df
    )

    qc_selection_df = safe_read_csv(
        QC_SELECTION_FILE
    )

    with pd.ExcelWriter(
        RESULTS_XLSX,
        engine="openpyxl",
    ) as writer:
        excel_presentation_copy(state_df).to_excel(
            writer,
            sheet_name="cases",
            index=False,
        )

        for method, sheet_name in [
            ("M1", "method1"),
            ("M2", "method2"),
            ("M3", "method3"),
        ]:
            if (
                not results_df.empty
                and "method"
                in results_df.columns
            ):
                table = results_df[
                    results_df[
                        "method"
                    ].eq(method)
                ].copy()
            else:
                table = pd.DataFrame()

            excel_presentation_copy(table).to_excel(
                writer,
                sheet_name=sheet_name,
                index=False,
            )

        excel_presentation_copy(comparison_df).to_excel(
            writer,
            sheet_name="comparison",
            index=False,
        )

        if not results_df.empty:
            qc_columns = [
                column
                for column in [
                    "case_id",
                    "breast",
                    "method",
                    "volume_sum_ok",
                    "percentage_sum",
                    "percentage_sum_ok",
                    "nipple_to_mask_mm",
                    "nipple_warning",
                ]
                if column
                in results_df.columns
            ]

            qc_df = results_df[
                qc_columns
            ].copy()
        else:
            qc_df = pd.DataFrame()

        excel_presentation_copy(qc_df).to_excel(
            writer,
            sheet_name="qc",
            index=False,
        )

        excel_presentation_copy(deformation_df).to_excel(
            writer,
            sheet_name="deformation",
            index=False,
        )

        excel_presentation_copy(qc_selection_df).to_excel(
            writer,
            sheet_name="random_qc",
            index=False,
        )

    format_excel_file(
        RESULTS_XLSX
    )


def save_all_state(
    state_df,
    results_df,
    deformation_df,
):
    atomic_csv(
        state_df,
        WORKFLOW_STATE_FILE,
    )

    atomic_csv(
        results_df,
        RESULTS_CSV,
    )

    atomic_csv(
        deformation_df,
        DEFORMATION_RESULTS_FILE,
    )

    save_excel_snapshot(
        state_df,
        results_df,
        deformation_df,
    )


def upsert_case_rows(
    dataframe,
    rows,
    case_id,
):
    new_rows = pd.DataFrame(
        rows
    )

    if dataframe.empty:
        return new_rows

    keep = (
        dataframe["case_id"]
        .astype(str)
        .ne(
            str(case_id)
        )
    )

    return pd.concat(
        [
            dataframe[
                keep
            ],
            new_rows,
        ],
        ignore_index=True,
    )


def upsert_one(
    dataframe,
    row,
    case_id,
):
    new_row = pd.DataFrame(
        [row]
    )

    if dataframe.empty:
        return new_row

    keep = (
        dataframe["case_id"]
        .astype(str)
        .ne(
            str(case_id)
        )
    )

    return pd.concat(
        [
            dataframe[
                keep
            ],
            new_row,
        ],
        ignore_index=True,
    )


def deformation_record(
    case_id,
    result,
    final_class,
    source,
    qc_path="",
):
    detections = result[
        "detections"
    ]

    scores = result[
        "scores"
    ]

    symmetry = result[
        "symmetry"
    ]

    def value(
        side,
        key,
    ):
        detection = detections[
            side
        ]

        if detection is None:
            return np.nan

        if key == "z":
            return int(
                detection["z"]
            )

        return detection[
            "line"
        ][key]

    return {
        "case_id":
            case_id,


        "auto_deformation_class":
            result["initial_class"],

        "deformation_class":
            final_class,

        "deformation_source":
            source,

        "deformation_excluded":
            final_class
            == "EXCLUDED",

        "L_deformation_score":
            scores["L"]["score"],

        "R_deformation_score":
            scores["R"]["score"],

        "bilateral_symmetry_score":
            symmetry["score"],

        "L_suspicious_z":
            value(
                "L",
                "z",
            ),

        "R_suspicious_z":
            value(
                "R",
                "z",
            ),

        "L_length_mm":
            value(
                "L",
                "length_mm",
            ),

        "R_length_mm":
            value(
                "R",
                "length_mm",
            ),

        "L_angle_deg":
            value(
                "L",
                "angle_deg",
            ),

        "R_angle_deg":
            value(
                "R",
                "angle_deg",
            ),

        "L_rmse_mm":
            value(
                "L",
                "rmse_mm",
            ),

        "R_rmse_mm":
            value(
                "R",
                "rmse_mm",
            ),

        "L_p95_mm":
            value(
                "L",
                "p95_mm",
            ),

        "R_p95_mm":
            value(
                "R",
                "p95_mm",
            ),

        "deformation_qc_image":
            str(qc_path),
    }


def _surface_mesh_ras(
    mask,
    affine,
    max_faces=45000,
):
    vertices, faces, _, _ = (
        measure.marching_cubes(
            mask.astype(
                np.uint8
            ),
            level=0.5,
        )
    )

    vertices_ras = ijk_to_ras(
        vertices,
        affine,
    )

    if len(faces) > max_faces:
        rng = np.random.default_rng(
            12345
        )

        keep = rng.choice(
            len(faces),
            size=max_faces,
            replace=False,
        )

        faces = faces[
            keep
        ]

    return (
        vertices_ras,
        faces.astype(
            np.int32
        ),
    )


def _face_quadrant_labels(
    vertices,
    faces,
    origin,
    medial_normal,
    superior_normal,
):
    centers = (
        vertices[
            faces
        ].mean(
            axis=1
        )
    )

    relative = (
        centers
        - np.asarray(
            origin
        )[None, :]
    )

    medial = (
        relative
        @ np.asarray(
            medial_normal
        )
    ) >= 0

    superior = (
        relative
        @ np.asarray(
            superior_normal
        )
    ) >= 0

    labels = np.empty(
        len(faces),
        dtype=object,
    )

    labels[
        superior & medial
    ] = "superior_medial"

    labels[
        superior & ~medial
    ] = "superior_lateral"

    labels[
        ~superior & medial
    ] = "inferior_medial"

    labels[
        ~superior & ~medial
    ] = "inferior_lateral"

    return labels


def _method_geometry_for_qc(
    method,
    side,
    breast_mask,
    affine,
    nipple,
):
    if method == "M1":
        origin = nipple

        medial = (
            np.array(
                [1.0, 0.0, 0.0]
            )
            if side == "L"
            else np.array(
                [-1.0, 0.0, 0.0]
            )
        )

        superior = np.array(
            [0.0, 0.0, 1.0]
        )

        points = {
            "nipple":
                nipple,
        }

        return (
            origin,
            medial,
            superior,
            points,
        )

    if method == "M2":
        ras = breast_points_ras(
            breast_mask,
            affine,
        )

        centroid = ras.mean(
            axis=0
        )

        _, medial, superior = (
            method2_axes(
                nipple,
                centroid,
                side,
            )
        )

        points = {
            "nipple":
                nipple,

            "centroid":
                centroid,
        }

        return (
            centroid,
            medial,
            superior,
            points,
        )

    if method == "M3":
        geometry = method3_geometry(
            breast_mask,
            affine,
            nipple,
            side,
        )

        points = {
            "nipple":
                nipple,

            "P":
                geometry["P"],

            "C":
                geometry["C"],
        }

        return (
            geometry["P"],
            geometry["medial_normal"],
            geometry["superior_normal"],
            points,
        )

    raise ValueError(
        f"Unknown method: {method}"
    )


def make_random_qc_html(
    case_id,
):
    mask, nifti_image = load_mask(
        case_id
    )

    nipples = load_nipples_ras(
        case_id
    )

    breasts, _ = split_breasts_native(
        mask,
        nifti_image.affine,
    )

    figure = go.Figure()

    for method in (
        "M1",
        "M2",
        "M3",
    ):
        visible_by_default = (
            method == "M1"
        )

        for side in (
            "L",
            "R",
        ):
            breast_mask = breasts[
                side
            ]

            (
                origin,
                medial,
                superior,
                points,
            ) = _method_geometry_for_qc(
                method,
                side,
                breast_mask,
                nifti_image.affine,
                nipples[side],
            )

            vertices, faces = (
                _surface_mesh_ras(
                    breast_mask,
                    nifti_image.affine,
                )
            )

            labels = (
                _face_quadrant_labels(
                    vertices,
                    faces,
                    origin,
                    medial,
                    superior,
                )
            )

            for quadrant in QUADRANTS:
                selected_faces = faces[
                    labels == quadrant
                ]

                if len(
                    selected_faces
                ) == 0:
                    continue

                figure.add_trace(
                    go.Mesh3d(
                        x=vertices[:, 0],
                        y=vertices[:, 1],
                        z=vertices[:, 2],
                        i=selected_faces[:, 0],
                        j=selected_faces[:, 1],
                        k=selected_faces[:, 2],
                        color=QUADRANT_COLORS[
                            quadrant
                        ],
                        opacity=0.62,
                        name=(
                            f"{method} | "
                            f"{side} | "
                            f"{quadrant}"
                        ),
                        legendgroup=method,
                        visible=visible_by_default,
                        hoverinfo="name",
                    )
                )

            for point_name, point in (
                points.items()
            ):
                figure.add_trace(
                    go.Scatter3d(
                        x=[
                            point[0]
                        ],
                        y=[
                            point[1]
                        ],
                        z=[
                            point[2]
                        ],
                        mode="markers",
                        marker={
                            "size": 5,
                            "color": "#111111",
                        },
                        name=(
                            f"{method} | "
                            f"{side} | "
                            f"{point_name}"
                        ),
                        legendgroup=method,
                        visible=visible_by_default,
                    )
                )

    buttons = []

    for method in (
        "M1",
        "M2",
        "M3",
    ):
        visibility = []

        for trace in figure.data:
            visibility.append(
                getattr(
                    trace,
                    "legendgroup",
                    None,
                )
                == method
            )

        buttons.append({
            "label":
                method,

            "method":
                "update",

            "args": [
                {
                    "visible":
                        visibility,
                },
                {
                    "title":
                        f"{case_id} — interactive QC — {method}",
                },
            ],
        })

    figure.update_layout(
        title=(
            f"{case_id} — "
            "interactive QC — M1"
        ),
        scene={
            "aspectmode":
                "data",

            "xaxis_title":
                "R/L (mm)",

            "yaxis_title":
                "A/P (mm)",

            "zaxis_title":
                "I/S (mm)",
        },
        updatemenus=[
            {
                "type":
                    "buttons",

                "direction":
                    "right",

                "buttons":
                    buttons,

                "x":
                    0.0,

                "y":
                    1.12,
            }
        ],
        margin={
            "l": 0,
            "r": 0,
            "b": 0,
            "t": 80,
        },
        showlegend=True,
    )

    html_path = (
        RANDOM_QC_DIR
        / f"{case_id}_QC.html"
    )

    div = plot(
        figure,
        include_plotlyjs="cdn",
        output_type="div",
        config={
            "responsive": True,
            "displaylogo": False,
        },
    )

    html_text = f"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>{case_id} — CLASSY-Breast QC</title>
<style>
body {{
    font-family: Arial, sans-serif;
    margin: 0;
}}
.info {{
    padding: 12px 18px;
    border-bottom: 1px solid #ddd;
}}
</style>
</head>
<body>
<div class="info">
<b>CLASSY-Breast — random interactive quality control</b><br>
Case: {case_id}<br>
Randomly selected from a completed block of 10 MRIs.<br>
Use the M1/M2/M3 buttons to compare the three geometric methods.
</div>
{div}
</body>
</html>
"""

    html_path.write_text(
        html_text,
        encoding="utf-8",
    )

    return html_path


def assign_processed_order(
    state_df,
    case_id,
):
    index = (
        state_df["case_id"]
        .astype(str)
        .eq(
            str(case_id)
        )
    )

    current = pd.to_numeric(
        state_df.loc[
            index,
            "processed_order",
        ],
        errors="coerce",
    )

    if current.notna().any():
        return state_df

    existing = pd.to_numeric(
        state_df[
            "processed_order"
        ],
        errors="coerce",
    )

    if existing.notna().any():
        next_order = int(
            existing.max()
        ) + 1
    else:
        next_order = 1

    state_df.loc[
        index,
        "processed_order",
    ] = next_order

    return state_df


def maybe_create_random_qc(
    state_df,
):
    completed = state_df[
        state_df[
            "pipeline_status"
        ].eq("DONE")
        &
        pd.to_numeric(
            state_df[
                "processed_order"
            ],
            errors="coerce",
        ).notna()
    ].copy()

    completed[
        "processed_order"
    ] = pd.to_numeric(
        completed[
            "processed_order"
        ],
        errors="coerce",
    ).astype(int)

    completed = completed.sort_values(
        "processed_order"
    )

    number_of_complete_batches = (
        len(completed)
        // 10
    )

    selections = safe_read_csv(
        QC_SELECTION_FILE
    )

    if selections.empty:
        selections = pd.DataFrame(
            columns=[
                "batch",
                "case_id",
                "html_path",
            ]
        )

    existing_batches = set(
        pd.to_numeric(
            selections["batch"],
            errors="coerce",
        )
        .dropna()
        .astype(int)
        .tolist()
    )

    new_reports = []

    for batch in range(
        1,
        number_of_complete_batches + 1,
    ):
        if batch in existing_batches:
            continue

        start = (
            batch - 1
        ) * 10

        block = completed.iloc[
            start:
            start + 10
        ]

        case_ids = (
            block["case_id"]
            .astype(str)
            .tolist()
        )

        rng = np.random.default_rng(
            QC_RANDOM_SEED
            + batch
        )

        selected_case = str(
            rng.choice(
                case_ids
            )
        )

        html_path = make_random_qc_html(
            selected_case
        )

        selections = pd.concat(
            [
                selections,
                pd.DataFrame(
                    [
                        {
                            "batch":
                                batch,

                            "case_id":
                                selected_case,

                            "html_path":
                                str(
                                    html_path
                                ),
                        }
                    ]
                ),
            ],
            ignore_index=True,
        )

        new_reports.append(
            (
                batch,
                selected_case,
                html_path,
            )
        )

    atomic_csv(
        selections,
        QC_SELECTION_FILE,
    )

    for (
        batch,
        selected_case,
        html_path,
    ) in new_reports:
        print()
        print(
            f"Random QC batch {batch}: "
            f"{selected_case}"
        )

        print(
            f"Saved HTML: {html_path}"
        )

        display(
            HTML(
                html_path.read_text(
                    encoding="utf-8"
                )
            )
        )

    return selections


## 8. Sequential case-by-case workflow

The existing case processing is unchanged: **segmentation → two axial deformation-QC images → PASS/EXCLUDED or optional Slicer inspection → nipple annotation → Methods 1–3 → numerical QC → Excel/CSV export**.

Output is shown directly in the notebook, without toggles, white widget panels or external widget downloads. Completed or excluded cases are skipped; the Slicer pause button saves the progress and stops the workflow.


In [ ]:

def process_one_case(
    case_id,
):
    state_df = build_workflow_state()
    results_df = safe_read_csv(
        RESULTS_CSV
    )
    deformation_df = safe_read_csv(
        DEFORMATION_RESULTS_FILE
    )

    index = (
        state_df["case_id"]
        .astype(str)
        .eq(
            str(case_id)
        )
    )

    if not index.any():
        raise RuntimeError(
            f"{case_id}: case not found in workflow state."
        )

    state_df.loc[
        index,
        "pipeline_status",
    ] = "PROCESSING"

    state_df.loc[
        index,
        "last_error",
    ] = ""

    save_all_state(
        state_df,
        results_df,
        deformation_df,
    )

    print()
    print("=" * 80)
    print(
        f"{case_id}"
    )
    print("=" * 80)

    try:
        # ----------------------------------------------------------
        # 1. Segmentation
        # ----------------------------------------------------------
        print()
        print(
            "1/5 — BREAST SEGMENTATION"
        )

        mask_path = segment_case(
            case_id,
            overwrite=False,
        )

        state_df.loc[
            index,
            "mask_path",
        ] = str(mask_path)

        mask_native, mask_nifti = load_mask(
            case_id
        )

        image_nifti = load_mri(
            case_id
        )

        image_native = np.asarray(
            image_nifti.dataobj,
            dtype=np.float32,
        ).squeeze()

        print(
            f"Mask: {mask_path}"
        )

        # ----------------------------------------------------------
        # 2. Deformation assessment
        # ----------------------------------------------------------
        print()
        print(
            "2/5 — DEFORMATION ASSESSMENT"
        )

        deformation_result = (
            analyse_deformation_case(
                case_id,
                mask_native,
                image_native,
                mask_nifti,
            )
        )

        automatic_class = (
            deformation_result[
                "initial_class"
            ]
        )

        if automatic_class == "REVIEW":
            display_deformation_qc(
                case_id=case_id,
                result=deformation_result,
                classification="REVIEW",
                source="automatic",
            )

            deformation_class = (
                ask_review_decision(
                    case_id
                )
            )

            deformation_source = (
                "manual_review"
            )

        else:
            deformation_class = (
                automatic_class
            )

            deformation_source = (
                "automatic"
            )

            # Informative display for every PASS and EXCLUDED case.
            display_deformation_qc(
                case_id=case_id,
                result=deformation_result,
                classification=deformation_class,
                source=deformation_source,
            )

        print(
            f"Deformation classification: {deformation_class}"
        )

        qc_image = save_deformation_qc(
            case_id=case_id,
            result=deformation_result,
            classification=deformation_class,
            source=deformation_source,
        )

        deformation_row = (
            deformation_record(
                case_id=case_id,
                result=deformation_result,
                final_class=deformation_class,
                source=deformation_source,
                qc_path=qc_image,
            )
        )

        deformation_df = upsert_one(
            deformation_df,
            deformation_row,
            case_id,
        )

        state_df.loc[
            index,
            "deformation_class",
        ] = deformation_class

        state_df.loc[
            index,
            "deformation_source",
        ] = deformation_source

        if deformation_class == "EXCLUDED":
            moved_mask = move_mask_to_deformed(
                case_id
            )

            state_df.loc[
                index,
                "mask_path",
            ] = str(
                moved_mask
            )

            state_df.loc[
                index,
                "pipeline_status",
            ] = "EXCLUDED"

            state_df.loc[
                index,
                "excluded",
            ] = True

            state_df.loc[
                index,
                "exclusion_reason",
            ] = "deformation"

            save_all_state(
                state_df,
                results_df,
                deformation_df,
            )

            print(
                f"{case_id}: excluded for deformation."
            )

            return {
                "status":
                    "EXCLUDED",

                "stop_requested":
                    False,
            }

        # ----------------------------------------------------------
        # 3. Nipple annotation
        # ----------------------------------------------------------
        print()
        print(
            "3/5 — NIPPLE ANNOTATION"
        )

        nipple_result = annotate_nipples(
            case_id
        )

        if (
            nipple_result["status"]
            == "PAUSE"
        ):
            state_df.loc[
                index,
                "pipeline_status",
            ] = "PAUSED_NIPPLE"

            save_all_state(
                state_df,
                results_df,
                deformation_df,
            )

            print(
                "Nipple annotation paused. "
                "All completed work has been saved."
            )

            return {
                "status":
                    "PAUSED_NIPPLE",

                "stop_requested":
                    True,
            }

        if (
            nipple_result["status"]
            == "EXCLUDED"
        ):
            state_df.loc[
                index,
                "pipeline_status",
            ] = "EXCLUDED"

            state_df.loc[
                index,
                "excluded",
            ] = True

            state_df.loc[
                index,
                "exclusion_reason",
            ] = (
                "nipple_not_annotatable"
            )

            save_all_state(
                state_df,
                results_df,
                deformation_df,
            )

            print(
                f"{case_id}: excluded because "
                "the nipples could not be annotated."
            )

            return {
                "status":
                    "EXCLUDED",

                "stop_requested":
                    False,
            }

        state_df.loc[
            index,
            "nipple_path",
        ] = nipple_result[
            "annotation"
        ]

        # ----------------------------------------------------------
        # 4. Three quadrant methods
        # ----------------------------------------------------------
        print()
        print(
            "4/5 — QUADRANT GEOMETRY"
        )

        print(
            "Computing Method 1..."
        )

        rows = method1(
            case_id
        )

        print(
            "Computing Method 2..."
        )

        rows += method2(
            case_id
        )

        print(
            "Computing Method 3..."
        )

        rows += method3(
            case_id
        )

        # ----------------------------------------------------------
        # 5. Numerical QC and export
        # ----------------------------------------------------------
        print()
        print(
            "5/5 — NUMERICAL QC AND EXPORT"
        )

        nipple_distance = nipple_qc(
            case_id
        )

        for row in rows:
            row.update(
                qc_method_row(
                    row
                )
            )

            row[
                "nipple_to_mask_mm"
            ] = nipple_distance[
                row["breast"]
            ]

            row[
                "nipple_warning"
            ] = (
                row[
                    "nipple_to_mask_mm"
                ]
                > 10.0
            )

        results_df = upsert_case_rows(
            results_df,
            rows,
            case_id,
        )

        state_df.loc[
            index,
            "pipeline_status",
        ] = "DONE"

        state_df.loc[
            index,
            "excluded",
        ] = False

        state_df.loc[
            index,
            "exclusion_reason",
        ] = ""

        state_df = assign_processed_order(
            state_df,
            case_id,
        )

        save_all_state(
            state_df,
            results_df,
            deformation_df,
        )

        # One random interactive QC report per block of 10 DONE cases.
        maybe_create_random_qc(
            state_df
        )

        # Refresh Excel after a new random QC selection has been recorded.
        save_excel_snapshot(
            state_df,
            results_df,
            deformation_df,
        )

        print()
        print(
            f"{case_id}: DONE"
        )

        print(
            f"Left nipple-to-mask distance: "
            f"{nipple_distance['L']:.2f} mm"
        )

        print(
            f"Right nipple-to-mask distance: "
            f"{nipple_distance['R']:.2f} mm"
        )

        display(
            pd.DataFrame(
                rows
            )
        )

        return {
            "status":
                "DONE",

            "stop_requested":
                False,
        }

    except KeyboardInterrupt:
        state_df.loc[
            index,
            "pipeline_status",
        ] = "PAUSED"

        save_all_state(
            state_df,
            results_df,
            deformation_df,
        )

        print(
            "Processing interrupted. "
            "The workflow state has been saved."
        )

        return {
            "status":
                "PAUSED",

            "stop_requested":
                True,
        }

    except Exception as error:
        state_df.loc[
            index,
            "pipeline_status",
        ] = "ERROR"

        state_df.loc[
            index,
            "last_error",
        ] = (
            f"{type(error).__name__}: "
            f"{error}"
        )

        save_all_state(
            state_df,
            results_df,
            deformation_df,
        )

        raise


def workflow_progress():
    current_state = build_workflow_state()

    eligible_ids = set(
        eligible_metadata[
            "patient_id"
        ].astype(str)
    )

    subset = current_state[
        current_state[
            "case_id"
        ].astype(str)
        .isin(
            eligible_ids
        )
    ]

    processed = int(
        subset[
            "pipeline_status"
        ]
        .isin(
            {
                "DONE",
                "EXCLUDED",
            }
        )
        .sum()
    )

    total = len(
        subset
    )

    return {
        "total":
            total,

        "processed":
            processed,

        "remaining":
            total - processed,
    }


def run_workflow():
    """Process eligible MRIs sequentially with ordinary notebook output.

    The two axial deformation-QC images, interactive decision prompts,
    Slicer annotation, persistent status, exports and pause/resume remain
    exactly as implemented by process_one_case(). No ipywidgets are used.
    """
    while True:
        current_state = build_workflow_state()
        eligible_ids = eligible_metadata["case_id"].astype(str).tolist()
        candidates = current_state[
            current_state["case_id"].astype(str).isin(eligible_ids)
            & ~current_state["pipeline_status"].isin({"DONE", "EXCLUDED"})
        ].copy()

        if candidates.empty:
            progress = workflow_progress()
            print("\n" + "=" * 70)
            print("WORKFLOW COMPLETE")
            print(f"Processed: {progress['processed']} / {progress['total']}")
            print("No metadata-eligible cases remain.")
            return

        candidates = candidates.sort_values(["case_id"])
        case = candidates.iloc[0]
        case_id = str(case["case_id"])
        progress = workflow_progress()

        print("\n" + "=" * 70, flush=True)
        print(f"{case_id} — PROCESSING", flush=True)
        print(
            f"Progress before: {progress['processed']} / {progress['total']} "
            f"processed; {progress['remaining']} remaining.",
            flush=True,
        )
        print("=" * 70, flush=True)

        result = process_one_case(case_id)
        updated = workflow_progress()
        print(f"\n{case_id} — {result['status']}", flush=True)
        print(
            f"Progress: {updated['processed']} / {updated['total']} "
            f"processed; {updated['remaining']} remaining.",
            flush=True,
        )

        if result["stop_requested"]:
            print("Workflow stopped. Run run_workflow() to resume.")
            return

        # Continue automatically; pause remains available in the Slicer dialog.


## 9. Start or resume the workflow

Run the cell below once after the configuration and function-definition cells have been executed.

The workflow automatically selects the next unfinished eligible MRI, processes it completely, saves its state and outputs, and continues until all cases are complete or the user explicitly pauses during nipple annotation.

Completed and excluded cases are skipped through the persistent CLASSY-Breast status files. Restarting the kernel does not require deleting previous results.


In [ ]:
run_workflow()
